In [4]:
import os
import gc
import re
import numpy as np
import pandas as pd
import torch
import importlib

import evaluate
importlib.reload(evaluate)

from varmax_osr import VarMaxOSR, DEFAULT_PAIR_MAP
from evaluate import choose_osr_calibration_splits, evaluate_multiple_osr_methods_on_run

/home/atrott/adamArchives/venvs/DNNs/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import pandas as pd

results_df = pd.read_csv("experiment_leaderboard.csv")
results_df

,run_name,save_dir,task,split_mode,num_classes,class_names,best_epoch,best_val_acc,best_val_macro_f1,test_loss,...,test_macro_f1,test_weighted_f1,test_n_samples,cache_len,batch_size,lr,weight_decay,lambda_center,epochs,seed
0,c16384_bs16_lr5e4_lc01_wd0_seed1,results_pa_finalist/11-55_04-22-26_c16384_bs16...,pa,closed,4,"['PA2', 'PA3', 'PA4', 'PA8']",6,0.997222,0.997222,0.078570,...,0.994444,0.994444,360,16384,16,0.0005,0.0,0.1,40,1
1,c16384_bs16_lr5e4_lc01_wd0_seed2,results_pa_finalist/11-57_04-22-26_c16384_bs16...,pa,closed,4,"['PA2', 'PA3', 'PA4', 'PA8']",13,0.997222,0.997222,0.093175,...,0.991666,0.991666,360,16384,16,0.0005,0.0,0.1,40,2


In [37]:
ota_runs_df = pd.DataFrame([
    {
        "run_name": "ota_btzb_ref_ent005_unkPA8_c16384_seed0",
        "save_dir": "results_pa_ota_btzb/12-01_05-01-26_ota_btzb_ref_ent005_unkPA8_c16384_seed0",
        "unknown_pa": "PA8",
    },
    {
        "run_name": "ota_btzb_ref_lr2e4_unkPA8_c16384_seed0",
        "save_dir": "results_pa_ota_btzb/12-07_05-01-26_ota_btzb_ref_lr2e4_unkPA8_c16384_seed0",
        "unknown_pa": "PA8",
    },
])

ota_runs_df

,run_name,save_dir,unknown_pa
0,ota_btzb_ref_ent005_unkPA8_c16384_seed0,results_pa_ota_btzb/12-01_05-01-26_ota_btzb_re...,PA8
1,ota_btzb_ref_lr2e4_unkPA8_c16384_seed0,results_pa_ota_btzb/12-07_05-01-26_ota_btzb_re...,PA8


In [40]:
def build_varmax_surrogate_aligned(payload, extras):
    return {
        "mode": "sweep",
        "calibration_mode": "surrogate_aligned",
        "calibration_known": payload.val_known,
        "pair_map": DEFAULT_PAIR_MAP,
        "surrogate_fit_frac": 0.50,
        "surrogate_guard_frac": 0.25,
        "surrogate_seed": 0,
        "known_floor_ratio": 0.95,
        "per_class_floor_ratio": 0.95,
        "unknown_recall_floor": 0.60,
        "top_k": 10,
    }

method_specs = [
    {
        "name": "VarMax-oracle-valopen-balanced",
        "factory": make_varmax,
        "calibration_builder": build_varmax_oracle_valopen_balanced,
    },
    {
        "name": "VarMax-surrogate-all",
        "factory": make_varmax,
        "calibration_builder": build_varmax_surrogate_all,
    },
    {
        "name": "VarMax-surrogate-aligned",
        "factory": make_varmax,
        "calibration_builder": build_varmax_surrogate_aligned,
    },
]

In [39]:
def build_varmax_oracle_valopen_balanced(payload, extras):
    known_cal, open_cal = choose_osr_calibration_splits(
        payload,
        extras,
        prefer_balanced=True,
    )
    return {
        "mode": "sweep",
        "calibration_mode": "oracle",
        "calibration_known": known_cal,
        "calibration_open": open_cal,
        "known_floor_ratio": 0.95,
        "per_class_floor_ratio": 0.95,
        "unknown_recall_floor": 0.60,
        "top_k": 10,
    }

method_specs = [
    {
        "name": "VarMax-oracle-valopen-balanced",
        "factory": make_varmax,
        "calibration_builder": build_varmax_oracle_valopen_balanced,
    }
]

In [42]:
import os
import gc
import pandas as pd
import torch

from varmax_osr import VarMaxOSR
from evaluate import evaluate_multiple_osr_methods_on_run, choose_osr_calibration_splits

DATA_ROOT = os.path.expanduser("~/Adam/varMax/PADataset/data")
device = "cuda" if torch.cuda.is_available() else "cpu"

def make_varmax():
    return VarMaxOSR(
        temperature=1.0,
        top2_threshold=None,
        top2_percentile=5.0,
        var_percentiles=(5.0, 95.0),
        energy_percentiles=(5.0, 95.0),
        use_energy=True,
        fit_on="predicted_class",
        min_samples_per_class=5,
    )

from varmax_osr import DEFAULT_PAIR_MAP

def build_varmax_surrogate_all(payload, extras):
    return {
        "mode": "sweep",
        "calibration_mode": "surrogate_all",
        "calibration_known": payload.val_known,
        "pair_map": DEFAULT_PAIR_MAP,
        "surrogate_fit_frac": 0.50,
        "surrogate_guard_frac": 0.25,
        "surrogate_seed": 0,
        "known_floor_ratio": 0.95,
        "per_class_floor_ratio": 0.95,
        "unknown_recall_floor": 0.60,
        "top_k": 10,
    }

method_specs = [
    {
        "name": "VarMax-oracle-valopen-balanced",
        "factory": make_varmax,
        "calibration_builder": build_varmax_oracle_valopen_balanced,
    },
    {
        "name": "VarMax-surrogate-all",
        "factory": make_varmax,
        "calibration_builder": build_varmax_surrogate_all,
    },
]



all_rows = []

for _, run_row in ota_runs_df.iterrows():
    print("Evaluating:", run_row["run_name"])

    result = evaluate_multiple_osr_methods_on_run(
        run_dir=run_row["save_dir"],
        method_specs=method_specs,
        checkpoint_tag="best_model",
        batch_size=64,
        num_workers=0,
        pin_memory=True,
        device=device,
        data_root=DATA_ROOT,
    )

    for method_name, fit_result in result["fitted"].items():
        all_rows.append({
            "run_name": run_row["run_name"],
            "unknown_pa": run_row["unknown_pa"],
            "method": method_name,
            "known_osr_macro_f1": fit_result["metrics"]["known_osr_macro_f1"],
            "unknown_recall": fit_result["metrics"]["unknown_recall"],
            "unknown_f1": fit_result["metrics"]["unknown_f1"],
            "osr_macro_f1": fit_result["metrics"]["osr_macro_f1"],
            "bias_delta": fit_result["metrics"]["bias_delta"],
            "unknown_auroc": fit_result["metrics"]["unknown_auroc"],
        })

    del result
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

ota_eval_df = pd.DataFrame(all_rows)
ota_eval_df.to_csv("ota_btzb_varmax_oracle_eval.csv", index=False)
display(ota_eval_df.sort_values(["run_name", "method"]))

Evaluating: ota_btzb_ref_ent005_unkPA8_c16384_seed0
Evaluating: ota_btzb_ref_lr2e4_unkPA8_c16384_seed0


,run_name,unknown_pa,method,known_osr_macro_f1,unknown_recall,unknown_f1,osr_macro_f1,bias_delta,unknown_auroc
0,ota_btzb_ref_ent005_unkPA8_c16384_seed0,PA8,VarMax-oracle-valopen-balanced,0.978310,0.853725,0.911260,0.874926,0.106555,0.992639
1,ota_btzb_ref_ent005_unkPA8_c16384_seed0,PA8,VarMax-surrogate-all,0.926009,0.893333,0.914676,0.857690,-0.012492,0.993881
2,ota_btzb_ref_lr2e4_unkPA8_c16384_seed0,PA8,VarMax-oracle-valopen-balanced,0.989679,0.910980,0.948550,0.922509,0.069549,0.987199
3,ota_btzb_ref_lr2e4_unkPA8_c16384_seed0,PA8,VarMax-surrogate-all,0.869797,0.940000,0.914188,0.840621,-0.171308,0.987694


In [ ]:
ota_runs_df = pd.DataFrame([
    {
        "run_name": "ota_btzb_ref_ent005_unkPA8_c16384_seed0",
        "save_dir": "results_pa_ota_btzb/12-01_05-01-26_ota_btzb_ref_ent005_unkPA8_c16384_seed0",
        "unknown_pa": "PA8",
    },
    {
        "run_name": "ota_btzb_ref_lr2e4_unkPA8_c16384_seed0",
        "save_dir": "results_pa_ota_btzb/12-07_05-01-26_ota_btzb_ref_lr2e4_unkPA8_c16384_seed0",
        "unknown_pa": "PA8",
    },
])

In [31]:
ota_runs_df = results_df.copy()

display(
    ota_runs_df[
        [
            "run_name",
            "save_dir",
            "unknown_pas",
            "known_pa_names",
            "source_name",
        ]
    ]
)

KeyError: "['unknown_pas', 'known_pa_names', 'source_name'] not in index"

In [ ]:
def make_varmax():
    return VarMaxOSR(
        temperature=1.0,
        top2_threshold=None,
        top2_percentile=5.0,
        var_percentiles=(5.0, 95.0),
        energy_percentiles=(5.0, 95.0),
        use_energy=True,
        fit_on="predicted_class",
        min_samples_per_class=5,
    )

def build_varmax_oracle_valopen_balanced(payload, extras):
    known_cal, open_cal = choose_osr_calibration_splits(
        payload,
        extras,
        prefer_balanced=True,
    )
    return {
        "mode": "sweep",
        "calibration_mode": "oracle",
        "calibration_known": known_cal,
        "calibration_open": open_cal,
        "known_floor_ratio": 0.95,
        "per_class_floor_ratio": 0.95,
        "unknown_recall_floor": 0.60,
        "top_k": 10,
    }

method_specs = [
    {
        "name": "VarMax-oracle-valopen-balanced",
        "factory": make_varmax,
        "calibration_builder": build_varmax_oracle_valopen_balanced,
    }
]

In [ ]:
DATA_ROOT = os.path.expanduser("~/Adam/varMax/PADataset/data")
device = "cuda" if torch.cuda.is_available() else "cpu"

all_rows = []

for _, run_row in ota_runs_df.iterrows():
    print("Evaluating:", run_row["run_name"])

    result = evaluate_multiple_osr_methods_on_run(
        run_dir=run_row["save_dir"],
        method_specs=method_specs,
        checkpoint_tag="best_model",
        batch_size=64,
        num_workers=0,
        pin_memory=True,
        device=device,
        data_root=DATA_ROOT,
    )

    unknown_pa = run_row["run_name"].split("_unk")[1].split("_")[0]

    for method_name, fit_result in result["fitted"].items():
        all_rows.append({
            "run_name": run_row["run_name"],
            "unknown_pa": unknown_pa,
            "method": method_name,
            "known_osr_macro_f1": fit_result["metrics"]["known_osr_macro_f1"],
            "unknown_recall": fit_result["metrics"]["unknown_recall"],
            "unknown_f1": fit_result["metrics"]["unknown_f1"],
            "osr_macro_f1": fit_result["metrics"]["osr_macro_f1"],
            "bias_delta": fit_result["metrics"]["bias_delta"],
            "unknown_auroc": fit_result["metrics"]["unknown_auroc"],
        })

    del result
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

ota_eval_df = pd.DataFrame(all_rows)
ota_eval_df.to_csv("ota_btzb_varmax_oracle_eval.csv", index=False)

display(ota_eval_df.sort_values(["run_name", "method"]))

In [ ]:
def build_varmax_surrogate_all(payload, extras):
    return {
        "mode": "sweep",
        "calibration_mode": "surrogate_all",
        "calibration_known": payload.val_known,
        "pair_map": DEFAULT_PAIR_MAP,
        "surrogate_fit_frac": 0.50,
        "surrogate_guard_frac": 0.25,
        "surrogate_seed": 0,
        "known_floor_ratio": 0.95,
        "per_class_floor_ratio": 0.95,
        "unknown_recall_floor": 0.60,
        "top_k": 10,
    }

method_specs = [
    {
        "name": "VarMax-oracle-valopen-balanced",
        "factory": make_varmax,
        "calibration_builder": build_varmax_oracle_valopen_balanced,
    },
    {
        "name": "VarMax-surrogate-all",
        "factory": make_varmax,
        "calibration_builder": build_varmax_surrogate_all,
    },
]

In [ ]:
DATA_ROOT = os.path.expanduser("~/Adam/varMax/PADataset/data")
device = "cuda" if torch.cuda.is_available() else "cpu"

all_rows = []

for _, run_row in ota_runs_df.iterrows():
    print("Evaluating:", run_row["run_name"])

    result = evaluate_multiple_osr_methods_on_run(
        run_dir=run_row["save_dir"],
        method_specs=method_specs,
        checkpoint_tag="best_model",
        batch_size=64,
        num_workers=0,
        pin_memory=True,
        device=device,
        data_root=DATA_ROOT,
    )

    unknown_pa = run_row["run_name"].split("_unk")[1].split("_")[0]

    for method_name, fit_result in result["fitted"].items():
        all_rows.append({
            "run_name": run_row["run_name"],
            "unknown_pa": unknown_pa,
            "method": method_name,
            "known_osr_macro_f1": fit_result["metrics"]["known_osr_macro_f1"],
            "unknown_recall": fit_result["metrics"]["unknown_recall"],
            "unknown_f1": fit_result["metrics"]["unknown_f1"],
            "osr_macro_f1": fit_result["metrics"]["osr_macro_f1"],
            "bias_delta": fit_result["metrics"]["bias_delta"],
            "unknown_auroc": fit_result["metrics"]["unknown_auroc"],
        })

    del result
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

ota_eval_df = pd.DataFrame(all_rows)
ota_eval_df.to_csv("ota_btzb_varmax_oracle_eval.csv", index=False)

display(ota_eval_df.sort_values(["run_name", "method"]))

In [17]:
DATA_ROOT = os.path.expanduser("~/Adam/varMax/PADataset/data")
device = "cuda" if torch.cuda.is_available() else "cpu"

refined_df = pd.read_csv("confmanifold_refined_leaderboard.csv")

refined_df["backbone_tag"] = refined_df["run_name"].str.replace(
    r"_unkPA\d+_c\d+_seed\d+$", "", regex=True
)

top_backbone_tags = [
    "ref_base_ent005",
    "ref_base_lr2e4",
    "ref_pms_drop040",
]

runs_to_eval = refined_df[refined_df["backbone_tag"].isin(top_backbone_tags)].copy()
runs_to_eval = runs_to_eval.sort_values(["backbone_tag", "run_name"]).reset_index(drop=True)

display(
    runs_to_eval[
        [
            "backbone_tag",
            "run_name",
            "save_dir",
            "unknown_pas",
            "test_dqn_proxy_softmax3",
            "test_dqn_proxy_expanded5",
            "test_known_macro_f1",
        ]
    ]
)

,backbone_tag,run_name,save_dir,unknown_pas,test_dqn_proxy_softmax3,test_dqn_proxy_expanded5,test_known_macro_f1
0,ref_base_ent005,ref_base_ent005_unkPA2_c16384_seed0,results_pa_confmanifold_refined/22-35_04-29-26...,['PA2'],0.969339,0.914038,0.996296
1,ref_base_ent005,ref_base_ent005_unkPA3_c16384_seed0,results_pa_confmanifold_refined/22-38_04-29-26...,['PA3'],0.987690,0.944207,0.988886
2,ref_base_ent005,ref_base_ent005_unkPA4_c16384_seed0,results_pa_confmanifold_refined/22-41_04-29-26...,['PA4'],0.946951,0.867263,0.963386
3,ref_base_ent005,ref_base_ent005_unkPA8_c16384_seed0,results_pa_confmanifold_refined/22-43_04-29-26...,['PA8'],0.988518,0.987758,0.985178
4,ref_base_lr2e4,ref_base_lr2e4_unkPA2_c16384_seed0,results_pa_confmanifold_refined/22-57_04-29-26...,['PA2'],0.980948,0.892330,0.996296
5,ref_base_lr2e4,ref_base_lr2e4_unkPA3_c16384_seed0,results_pa_confmanifold_refined/23-00_04-29-26...,['PA3'],0.992160,0.942151,0.996296
6,ref_base_lr2e4,ref_base_lr2e4_unkPA4_c16384_seed0,results_pa_confmanifold_refined/23-04_04-29-26...,['PA4'],0.980775,0.923598,0.992592
7,ref_base_lr2e4,ref_base_lr2e4_unkPA8_c16384_seed0,results_pa_confmanifold_refined/23-07_04-29-26...,['PA8'],0.988998,0.952102,0.985178
8,ref_pms_drop040,ref_pms_drop040_unkPA2_c16384_seed0,results_pa_confmanifold_refined/23-45_04-29-26...,['PA2'],0.955565,0.901787,0.996296
9,ref_pms_drop040,ref_pms_drop040_unkPA3_c16384_seed0,results_pa_confmanifold_refined/23-49_04-29-26...,['PA3'],0.996146,0.948451,0.996296


In [18]:
def make_varmax():
    return VarMaxOSR(
        temperature=1.0,
        top2_threshold=None,
        top2_percentile=5.0,
        var_percentiles=(5.0, 95.0),
        energy_percentiles=(5.0, 95.0),
        use_energy=True,
        fit_on="predicted_class",
        min_samples_per_class=5,
    )

In [19]:
def build_varmax_oracle_valopen_balanced(payload, extras):
    known_cal, open_cal = choose_osr_calibration_splits(
        payload,
        extras,
        prefer_balanced=True,
    )
    return {
        "mode": "sweep",
        "calibration_mode": "oracle",
        "calibration_known": known_cal,
        "calibration_open": open_cal,
        "known_floor_ratio": 0.95,
        "per_class_floor_ratio": 0.95,
        "unknown_recall_floor": 0.60,
        "top_k": 10,
    }


def build_varmax_surrogate_all(payload, extras):
    return {
        "mode": "sweep",
        "calibration_mode": "surrogate_all",
        "calibration_known": payload.val_known,
        "pair_map": DEFAULT_PAIR_MAP,
        "surrogate_fit_frac": 0.50,
        "surrogate_guard_frac": 0.25,
        "surrogate_seed": 0,
        "known_floor_ratio": 0.95,
        "per_class_floor_ratio": 0.95,
        "unknown_recall_floor": 0.60,
        "top_k": 10,
    }


def build_varmax_surrogate_aligned(payload, extras):
    return {
        "mode": "sweep",
        "calibration_mode": "surrogate_aligned",
        "calibration_known": payload.val_known,
        "pair_map": DEFAULT_PAIR_MAP,
        "surrogate_fit_frac": 0.50,
        "surrogate_guard_frac": 0.25,
        "surrogate_seed": 0,
        "known_floor_ratio": 0.95,
        "per_class_floor_ratio": 0.95,
        "unknown_recall_floor": 0.60,
        "top_k": 10,
    }


method_specs = [
    {
        "name": "VarMax-oracle-valopen-balanced",
        "factory": make_varmax,
        "calibration_builder": build_varmax_oracle_valopen_balanced,
    },
    {
        "name": "VarMax-surrogate-all",
        "factory": make_varmax,
        "calibration_builder": build_varmax_surrogate_all,
    },
    {
        "name": "VarMax-surrogate-aligned",
        "factory": make_varmax,
        "calibration_builder": build_varmax_surrogate_aligned,
    },
]

In [21]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# Assumes varmax_final_df already exists in memory
# ------------------------------------------------------------------

OUTDIR = os.path.expanduser("~/Adam/varMax/PADataset/results/presentation/varmax_figs")
os.makedirs(OUTDIR, exist_ok=True)

df = varmax_final_df.copy()

# ------------------------------------------------------------
# Friendly labels / ordering
# ------------------------------------------------------------
method_order = [
    "VarMax-oracle-valopen-balanced",
    "VarMax-surrogate-all",
    "VarMax-surrogate-aligned",
]
method_label = {
    "VarMax-oracle-valopen-balanced": "Oracle",
    "VarMax-surrogate-all": "Surrogate-all",
    "VarMax-surrogate-aligned": "Surrogate-aligned",
}
unknown_order = ["PA2", "PA3", "PA4", "PA8"]
backbone_order = ["ref_base_ent005", "ref_base_lr2e4", "ref_pms_drop040"]

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def savefig(name):
    path = os.path.join(OUTDIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.close()
    print(f"saved: {path}")

def grouped_bar_from_summary(summary_df, x_col, series_cols, title, ylabel, filename,
                             x_order=None, legend_labels=None, rotation=0):
    plot_df = summary_df.copy()
    if x_order is not None:
        plot_df[x_col] = pd.Categorical(plot_df[x_col], categories=x_order, ordered=True)
        plot_df = plot_df.sort_values(x_col)

    x = np.arange(len(plot_df))
    n = len(series_cols)
    width = 0.8 / n

    plt.figure(figsize=(10, 5))
    for i, col in enumerate(series_cols):
        label = legend_labels[i] if legend_labels is not None else col
        plt.bar(x + (i - (n - 1) / 2) * width, plot_df[col].values, width=width, label=label)

    plt.xticks(x, plot_df[x_col].astype(str).tolist(), rotation=rotation)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.ylim(0, 1.05)
    plt.grid(axis="y", alpha=0.25)
    plt.legend(frameon=False)
    savefig(filename)

def draw_heatmap(mat_df, title, filename, vmin=0.0, vmax=1.0, annotate=True):
    plt.figure(figsize=(8, 4.8))
    plt.imshow(mat_df.values, aspect="auto", vmin=vmin, vmax=vmax)
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.xticks(np.arange(mat_df.shape[1]), mat_df.columns, rotation=20, ha="right")
    plt.yticks(np.arange(mat_df.shape[0]), mat_df.index)
    plt.title(title)

    if annotate:
        for i in range(mat_df.shape[0]):
            for j in range(mat_df.shape[1]):
                val = mat_df.iloc[i, j]
                if pd.notna(val):
                    plt.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=9)

    savefig(filename)

# ------------------------------------------------------------
# 1) Regime-level summary figure
# ------------------------------------------------------------
protocol_summary_df = (
    df.groupby(["method"], as_index=False)
      .agg(
          mean_known_osr_macro_f1=("known_osr_macro_f1", "mean"),
          min_known_osr_macro_f1=("known_osr_macro_f1", "min"),
          mean_unknown_recall=("unknown_recall", "mean"),
          min_unknown_recall=("unknown_recall", "min"),
          mean_unknown_f1=("unknown_f1", "mean"),
          min_unknown_f1=("unknown_f1", "min"),
          mean_osr_macro_f1=("osr_macro_f1", "mean"),
          min_osr_macro_f1=("osr_macro_f1", "min"),
          mean_bias_delta=("bias_delta", "mean"),
          max_abs_bias_delta=("bias_delta", lambda s: float(np.max(np.abs(s)))),
          mean_unknown_auroc=("unknown_auroc", "mean"),
          n_infeasible=("no_feasible_solution", lambda s: int(np.sum(np.asarray(s).astype(bool)))),
      )
)

protocol_summary_df["method_label"] = protocol_summary_df["method"].map(method_label)

grouped_bar_from_summary(
    protocol_summary_df,
    x_col="method_label",
    series_cols=[
        "mean_osr_macro_f1",
        "mean_unknown_f1",
        "mean_unknown_recall",
        "mean_unknown_auroc",
    ],
    legend_labels=[
        "Mean OSR Macro-F1",
        "Mean Unknown F1",
        "Mean Unknown Recall",
        "Mean Unknown AUROC",
    ],
    title="VarMax Regime-Level Results",
    ylabel="Score",
    filename="regime_summary_metrics.png",
    x_order=[method_label[m] for m in method_order],
)

# ------------------------------------------------------------
# 2) Backbone x method heatmap
# ------------------------------------------------------------
heat_df = (
    df.groupby(["backbone_tag", "method"], as_index=False)["osr_macro_f1"]
      .mean()
      .pivot(index="backbone_tag", columns="method", values="osr_macro_f1")
)

heat_df = heat_df.reindex(index=backbone_order, columns=method_order)
heat_df.columns = [method_label[c] for c in heat_df.columns]

draw_heatmap(
    heat_df,
    title="Mean OSR Macro-F1 by Backbone and Calibration Regime",
    filename="backbone_method_heatmap_osr_macro_f1.png",
)

# ------------------------------------------------------------
# 3) Per-unknown breakdown by regime: OSR macro-F1
# ------------------------------------------------------------
per_unknown_osr = (
    df.groupby(["unknown_pa", "method"], as_index=False)["osr_macro_f1"]
      .mean()
      .pivot(index="unknown_pa", columns="method", values="osr_macro_f1")
      .reindex(index=unknown_order, columns=method_order)
)

plt.figure(figsize=(10, 5))
x = np.arange(len(per_unknown_osr.index))
width = 0.24
for i, m in enumerate(method_order):
    plt.bar(x + (i - 1) * width, per_unknown_osr[m].values, width=width, label=method_label[m])

plt.xticks(x, per_unknown_osr.index.tolist())
plt.ylabel("Mean OSR Macro-F1")
plt.title("OSR Macro-F1 by Held-Out Unknown PA")
plt.ylim(0, 1.05)
plt.grid(axis="y", alpha=0.25)
plt.legend(frameon=False)
savefig("per_unknown_osr_macro_f1.png")

# ------------------------------------------------------------
# 4) Per-unknown breakdown by regime: Unknown F1
# ------------------------------------------------------------
per_unknown_f1 = (
    df.groupby(["unknown_pa", "method"], as_index=False)["unknown_f1"]
      .mean()
      .pivot(index="unknown_pa", columns="method", values="unknown_f1")
      .reindex(index=unknown_order, columns=method_order)
)

plt.figure(figsize=(10, 5))
x = np.arange(len(per_unknown_f1.index))
width = 0.24
for i, m in enumerate(method_order):
    plt.bar(x + (i - 1) * width, per_unknown_f1[m].values, width=width, label=method_label[m])

plt.xticks(x, per_unknown_f1.index.tolist())
plt.ylabel("Mean Unknown F1")
plt.title("Unknown F1 by Held-Out Unknown PA")
plt.ylim(0, 1.05)
plt.grid(axis="y", alpha=0.25)
plt.legend(frameon=False)
savefig("per_unknown_unknown_f1.png")

# ------------------------------------------------------------
# 5) Deployable winner breakdown
# ------------------------------------------------------------
winner_df = df[
    (df["backbone_tag"] == "ref_base_ent005") &
    (df["method"] == "VarMax-surrogate-all")
].copy()

winner_df["unknown_pa"] = pd.Categorical(winner_df["unknown_pa"], categories=unknown_order, ordered=True)
winner_df = winner_df.sort_values("unknown_pa")

metrics = ["osr_macro_f1", "unknown_f1", "unknown_recall", "known_osr_macro_f1"]
metric_labels = ["OSR Macro-F1", "Unknown F1", "Unknown Recall", "Known OSR Macro-F1"]

plt.figure(figsize=(11, 5.5))
x = np.arange(len(winner_df))
width = 0.18

for i, metric in enumerate(metrics):
    plt.bar(x + (i - 1.5) * width, winner_df[metric].values, width=width, label=metric_labels[i])

plt.xticks(x, winner_df["unknown_pa"].astype(str).tolist())
plt.ylabel("Score")
plt.title("Best Deployable VarMax Setup: ref_base_ent005 + Surrogate-all")
plt.ylim(0, 1.05)
plt.grid(axis="y", alpha=0.25)
plt.legend(frameon=False, ncol=2)
savefig("deployable_winner_breakdown.png")

# ------------------------------------------------------------
# 6) Compare best backbones for surrogate-all vs aligned
# ------------------------------------------------------------
main_legit_df = (
    df[df["method"] == "VarMax-surrogate-all"]
    .groupby("backbone_tag", as_index=False)
    .agg(
        mean_osr_macro_f1=("osr_macro_f1", "mean"),
        min_osr_macro_f1=("osr_macro_f1", "min"),
        mean_unknown_f1=("unknown_f1", "mean"),
        min_unknown_f1=("unknown_f1", "min"),
        mean_unknown_recall=("unknown_recall", "mean"),
        max_abs_bias_delta=("bias_delta", lambda s: float(np.max(np.abs(s)))),
    )
)

aligned_df = (
    df[df["method"] == "VarMax-surrogate-aligned"]
    .groupby("backbone_tag", as_index=False)
    .agg(
        mean_osr_macro_f1=("osr_macro_f1", "mean"),
        min_osr_macro_f1=("osr_macro_f1", "min"),
        mean_unknown_f1=("unknown_f1", "mean"),
        min_unknown_f1=("unknown_f1", "min"),
        mean_unknown_recall=("unknown_recall", "mean"),
        max_abs_bias_delta=("bias_delta", lambda s: float(np.max(np.abs(s)))),
    )
)

main_legit_df["regime"] = "Surrogate-all"
aligned_df["regime"] = "Surrogate-aligned"

compare_df = pd.concat([main_legit_df, aligned_df], ignore_index=True)
compare_df["backbone_tag"] = pd.Categorical(compare_df["backbone_tag"], categories=backbone_order, ordered=True)
compare_df = compare_df.sort_values(["regime", "backbone_tag"])

# plot mean_osr_macro_f1 only, cleaner for slide use
plot_df = compare_df.pivot(index="backbone_tag", columns="regime", values="mean_osr_macro_f1")
plot_df = plot_df.reindex(index=backbone_order, columns=["Surrogate-all", "Surrogate-aligned"])

plt.figure(figsize=(9, 5))
x = np.arange(len(plot_df.index))
width = 0.34
plt.bar(x - width/2, plot_df["Surrogate-all"].values, width=width, label="Surrogate-all")
plt.bar(x + width/2, plot_df["Surrogate-aligned"].values, width=width, label="Surrogate-aligned")
plt.xticks(x, plot_df.index.tolist())
plt.ylabel("Mean OSR Macro-F1")
plt.title("Backbone Comparison: Practical vs Structure-Aware Surrogate Calibration")
plt.ylim(0, 1.05)
plt.grid(axis="y", alpha=0.25)
plt.legend(frameon=False)
savefig("deployable_vs_structureaware_backbones.png")

# ------------------------------------------------------------
# 7) Bias delta distribution by method
# ------------------------------------------------------------
bias_groups = [df[df["method"] == m]["bias_delta"].values for m in method_order]

plt.figure(figsize=(9, 5))
plt.boxplot(
    bias_groups,
    labels=[method_label[m] for m in method_order],
    vert=True,
    patch_artist=False,
)
plt.axhline(0.0, linestyle="--", linewidth=1)
plt.ylabel("Bias Delta (known_accept_rate - unknown_recall)")
plt.title("Bias Delta by Calibration Regime")
plt.grid(axis="y", alpha=0.25)
savefig("bias_delta_by_method.png")

print("\nDone. Figures written to:")
print(OUTDIR)

saved: /home/atrott/Adam/varMax/PADataset/results/presentation/varmax_figs/regime_summary_metrics.png
saved: /home/atrott/Adam/varMax/PADataset/results/presentation/varmax_figs/backbone_method_heatmap_osr_macro_f1.png
saved: /home/atrott/Adam/varMax/PADataset/results/presentation/varmax_figs/per_unknown_osr_macro_f1.png
saved: /home/atrott/Adam/varMax/PADataset/results/presentation/varmax_figs/per_unknown_unknown_f1.png
saved: /home/atrott/Adam/varMax/PADataset/results/presentation/varmax_figs/deployable_winner_breakdown.png
saved: /home/atrott/Adam/varMax/PADataset/results/presentation/varmax_figs/deployable_vs_structureaware_backbones.png
saved: /home/atrott/Adam/varMax/PADataset/results/presentation/varmax_figs/bias_delta_by_method.png

Done. Figures written to:
/home/atrott/Adam/varMax/PADataset/results/presentation/varmax_figs


/tmp/ipykernel_3007735/104430241.py:280: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(


In [20]:
run_row = runs_to_eval.iloc[0]

result = evaluate_multiple_osr_methods_on_run(
    run_dir=run_row["save_dir"],
    method_specs=method_specs,
    checkpoint_tag="best_model",
    batch_size=64,
    num_workers=0,
    pin_memory=True,
    device=device,
    data_root=DATA_ROOT,
)

print(result["handle"].run_name)
print(result["payload"].meta)
print("val_known:", result["payload"].val_known.logits.shape)
print("test_known:", result["payload"].test_known.logits.shape)
print("test_open:", result["payload"].test_open.logits.shape)
print("val_open:", None if result["extras"]["val_open"] is None else result["extras"]["val_open"].logits.shape)
print("val_known_balanced:", None if result["extras"]["val_known_balanced"] is None else result["extras"]["val_known_balanced"].logits.shape)
print("val_open_balanced:", None if result["extras"]["val_open_balanced"] is None else result["extras"]["val_open_balanced"].logits.shape)

display(
    result["df"][
        [
            "method",
            "known_osr_macro_f1",
            "unknown_recall",
            "unknown_f1",
            "osr_macro_f1",
            "bias_delta",
            "unknown_auroc",
        ]
    ]
)

KeyboardInterrupt: 

In [11]:
all_rows = []

for i, (_, run_row) in enumerate(runs_to_eval.iterrows(), start=1):
    print(f"[{i}/{len(runs_to_eval)}] {run_row['run_name']}")

    result = evaluate_multiple_osr_methods_on_run(
        run_dir=run_row["save_dir"],
        method_specs=method_specs,
        checkpoint_tag="best_model",
        batch_size=64,
        num_workers=0,
        pin_memory=True,
        device=device,
        data_root=DATA_ROOT,
    )

    unknown_pa = run_row["run_name"].split("_unk")[1].split("_")[0]

    for method_name, fit_result in result["fitted"].items():
        params = fit_result["params"]
        best_sweep = params.get("best_sweep_result", None)

        all_rows.append({
            "backbone_tag": run_row["backbone_tag"],
            "run_name": run_row["run_name"],
            "run_dir": run_row["save_dir"],
            "unknown_pa": unknown_pa,
            "method": method_name,

            "n_val_known_balanced": (
                0 if result["extras"]["val_known_balanced"] is None
                else len(result["extras"]["val_known_balanced"].y_true)
            ),
            "n_val_open_balanced": (
                0 if result["extras"]["val_open_balanced"] is None
                else len(result["extras"]["val_open_balanced"].y_true)
            ),

            "calibration_mode": params.get("calibration_mode"),
            "no_feasible_solution": params.get("no_feasible_solution"),

            "best_top2_threshold": None if best_sweep is None else best_sweep.get("top2_threshold"),
            "best_var_percentiles": None if best_sweep is None else str(best_sweep.get("var_percentiles")),
            "best_energy_percentiles": None if best_sweep is None else str(best_sweep.get("energy_percentiles")),
            "best_regime": None if best_sweep is None else best_sweep.get("regime"),
            "best_spec_name": None if best_sweep is None else best_sweep.get("spec_name"),
            "best_heldout_class_name": None if best_sweep is None else best_sweep.get("heldout_class_name"),
            "best_is_structure_aligned": None if best_sweep is None else best_sweep.get("is_structure_aligned"),

            **fit_result["metrics"],
        })

    del result
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

varmax_final_df = pd.DataFrame(all_rows)
varmax_final_df.to_csv("varmax_final_setup_eval.csv", index=False)

display(
    varmax_final_df[
        [
            "backbone_tag",
            "unknown_pa",
            "method",
            "calibration_mode",
            "known_osr_macro_f1",
            "unknown_recall",
            "unknown_f1",
            "osr_macro_f1",
            "bias_delta",
            "unknown_auroc",
            "no_feasible_solution",
            "best_regime",
            "best_heldout_class_name",
            "best_is_structure_aligned",
        ]
    ].sort_values(["backbone_tag", "method", "unknown_pa"])
)

[1/12] ref_base_ent005_unkPA2_c16384_seed0
[2/12] ref_base_ent005_unkPA3_c16384_seed0
[3/12] ref_base_ent005_unkPA4_c16384_seed0
[4/12] ref_base_ent005_unkPA8_c16384_seed0
[5/12] ref_base_lr2e4_unkPA2_c16384_seed0
[6/12] ref_base_lr2e4_unkPA3_c16384_seed0
[7/12] ref_base_lr2e4_unkPA4_c16384_seed0
[8/12] ref_base_lr2e4_unkPA8_c16384_seed0
[9/12] ref_pms_drop040_unkPA2_c16384_seed0
[10/12] ref_pms_drop040_unkPA3_c16384_seed0
[11/12] ref_pms_drop040_unkPA4_c16384_seed0
[12/12] ref_pms_drop040_unkPA8_c16384_seed0


,backbone_tag,unknown_pa,method,calibration_mode,known_osr_macro_f1,unknown_recall,unknown_f1,osr_macro_f1,bias_delta,unknown_auroc,no_feasible_solution,best_regime,best_heldout_class_name,best_is_structure_aligned
0,ref_base_ent005,PA2,VarMax-oracle-valopen-balanced,oracle,0.902622,0.505882,0.634686,0.687658,0.327451,0.986006,True,oracle,NaN,None
3,ref_base_ent005,PA3,VarMax-oracle-valopen-balanced,oracle,0.960716,0.890196,0.924644,0.887624,0.043137,0.968475,False,oracle,NaN,None
6,ref_base_ent005,PA4,VarMax-oracle-valopen-balanced,oracle,0.971823,0.968627,0.977250,0.952065,0.005447,0.966376,False,oracle,NaN,None
9,ref_base_ent005,PA8,VarMax-oracle-valopen-balanced,oracle,0.965570,0.949020,0.964143,0.934001,0.013943,0.987778,False,oracle,NaN,None
2,ref_base_ent005,PA2,VarMax-surrogate-aligned,surrogate_aligned,0.973221,0.237255,0.376361,0.641127,0.718301,0.976943,True,aligned,PA8,True
5,ref_base_ent005,PA3,VarMax-surrogate-aligned,surrogate_aligned,0.950311,0.878431,0.913354,0.871850,0.036383,0.963420,True,aligned,PA4,True
8,ref_base_ent005,PA4,VarMax-surrogate-aligned,surrogate_aligned,0.933728,0.968627,0.963902,0.920489,-0.046405,0.965643,True,aligned,PA3,True
11,ref_base_ent005,PA8,VarMax-surrogate-aligned,surrogate_aligned,0.929672,0.958824,0.953216,0.910244,-0.058824,0.985447,True,aligned,PA2,True
1,ref_base_ent005,PA2,VarMax-surrogate-all,surrogate_all,0.942303,0.341176,0.489451,0.660645,0.558824,0.974285,True,mismatched,PA3,False
4,ref_base_ent005,PA3,VarMax-surrogate-all,surrogate_all,0.912483,0.925490,0.920976,0.867893,-0.084749,0.970080,True,mismatched,PA2,False


In [12]:
summary_df = (
    varmax_final_df
    .groupby(["backbone_tag", "method"], as_index=False)
    .agg(
        mean_known_osr_macro_f1=("known_osr_macro_f1", "mean"),
        min_known_osr_macro_f1=("known_osr_macro_f1", "min"),
        mean_unknown_recall=("unknown_recall", "mean"),
        min_unknown_recall=("unknown_recall", "min"),
        mean_unknown_f1=("unknown_f1", "mean"),
        min_unknown_f1=("unknown_f1", "min"),
        mean_osr_macro_f1=("osr_macro_f1", "mean"),
        min_osr_macro_f1=("osr_macro_f1", "min"),
        mean_bias_delta=("bias_delta", "mean"),
        max_abs_bias_delta=("bias_delta", lambda s: float(np.max(np.abs(s)))),
        mean_unknown_auroc=("unknown_auroc", "mean"),
        n_infeasible=("no_feasible_solution", lambda s: int(np.sum(np.asarray(s).astype(bool)))),
    )
    .sort_values(
        ["mean_osr_macro_f1", "mean_unknown_f1", "mean_unknown_recall"],
        ascending=False,
    )
)

display(summary_df)

,backbone_tag,method,mean_known_osr_macro_f1,min_known_osr_macro_f1,mean_unknown_recall,min_unknown_recall,mean_unknown_f1,min_unknown_f1,mean_osr_macro_f1,min_osr_macro_f1,mean_bias_delta,max_abs_bias_delta,mean_unknown_auroc,n_infeasible
6,ref_pms_drop040,VarMax-oracle-valopen-balanced,0.954421,0.876868,0.802451,0.286275,0.838548,0.409537,0.873620,0.603651,0.129031,0.502614,0.952490,1
0,ref_base_ent005,VarMax-oracle-valopen-balanced,0.950183,0.902622,0.828431,0.505882,0.875181,0.634686,0.865337,0.687658,0.097495,0.327451,0.977159,1
2,ref_base_ent005,VarMax-surrogate-all,0.929547,0.912483,0.798529,0.341176,0.831886,0.489451,0.839818,0.660645,0.092211,0.558824,0.973863,4
1,ref_base_ent005,VarMax-surrogate-aligned,0.946733,0.929672,0.760784,0.237255,0.801708,0.376361,0.835928,0.641127,0.162364,0.718301,0.972863,4
3,ref_base_lr2e4,VarMax-oracle-valopen-balanced,0.935660,0.869061,0.711275,0.443137,0.787848,0.567839,0.808388,0.652251,0.186874,0.445752,0.972260,2
4,ref_base_lr2e4,VarMax-surrogate-aligned,0.903994,0.853555,0.729902,0.492157,0.789078,0.625935,0.791083,0.702175,0.108987,0.355991,0.974784,4
8,ref_pms_drop040,VarMax-surrogate-all,0.928944,0.898830,0.662255,0.154902,0.712335,0.250396,0.788508,0.570430,0.223856,0.689542,0.951143,4
5,ref_base_lr2e4,VarMax-surrogate-all,0.904242,0.853555,0.681863,0.300000,0.742008,0.432815,0.771503,0.625804,0.159804,0.537037,0.974210,4
7,ref_pms_drop040,VarMax-surrogate-aligned,0.930234,0.878716,0.567157,0.209804,0.662274,0.317979,0.748574,0.578178,0.322658,0.582789,0.958017,4


In [13]:
protocol_summary_df = (
    varmax_final_df
    .groupby(["method"], as_index=False)
    .agg(
        mean_known_osr_macro_f1=("known_osr_macro_f1", "mean"),
        min_known_osr_macro_f1=("known_osr_macro_f1", "min"),
        mean_unknown_recall=("unknown_recall", "mean"),
        min_unknown_recall=("unknown_recall", "min"),
        mean_unknown_f1=("unknown_f1", "mean"),
        min_unknown_f1=("unknown_f1", "min"),
        mean_osr_macro_f1=("osr_macro_f1", "mean"),
        min_osr_macro_f1=("osr_macro_f1", "min"),
        mean_bias_delta=("bias_delta", "mean"),
        max_abs_bias_delta=("bias_delta", lambda s: float(np.max(np.abs(s)))),
        mean_unknown_auroc=("unknown_auroc", "mean"),
        n_infeasible=("no_feasible_solution", lambda s: int(np.sum(np.asarray(s).astype(bool)))),
    )
    .sort_values(
        ["mean_osr_macro_f1", "mean_unknown_f1", "mean_unknown_recall"],
        ascending=False,
    )
)

display(protocol_summary_df)

,method,mean_known_osr_macro_f1,min_known_osr_macro_f1,mean_unknown_recall,min_unknown_recall,mean_unknown_f1,min_unknown_f1,mean_osr_macro_f1,min_osr_macro_f1,mean_bias_delta,max_abs_bias_delta,mean_unknown_auroc,n_infeasible
0,VarMax-oracle-valopen-balanced,0.946755,0.869061,0.780719,0.286275,0.833859,0.409537,0.849115,0.603651,0.137800,0.502614,0.967303,4
2,VarMax-surrogate-all,0.920911,0.853555,0.714216,0.154902,0.762076,0.250396,0.799943,0.570430,0.158624,0.689542,0.966406,12
1,VarMax-surrogate-aligned,0.926987,0.853555,0.685948,0.209804,0.751020,0.317979,0.791862,0.578178,0.198003,0.718301,0.968555,12


In [14]:
main_legit_df = (
    varmax_final_df[varmax_final_df["method"] == "VarMax-surrogate-all"]
    .groupby("backbone_tag", as_index=False)
    .agg(
        mean_osr_macro_f1=("osr_macro_f1", "mean"),
        min_osr_macro_f1=("osr_macro_f1", "min"),
        mean_unknown_f1=("unknown_f1", "mean"),
        min_unknown_f1=("unknown_f1", "min"),
        mean_unknown_recall=("unknown_recall", "mean"),
        max_abs_bias_delta=("bias_delta", lambda s: float(np.max(np.abs(s)))),
    )
    .sort_values(
        ["mean_osr_macro_f1", "min_osr_macro_f1", "mean_unknown_f1"],
        ascending=False,
    )
)

aligned_df = (
    varmax_final_df[varmax_final_df["method"] == "VarMax-surrogate-aligned"]
    .groupby("backbone_tag", as_index=False)
    .agg(
        mean_osr_macro_f1=("osr_macro_f1", "mean"),
        min_osr_macro_f1=("osr_macro_f1", "min"),
        mean_unknown_f1=("unknown_f1", "mean"),
        min_unknown_f1=("unknown_f1", "min"),
        mean_unknown_recall=("unknown_recall", "mean"),
        max_abs_bias_delta=("bias_delta", lambda s: float(np.max(np.abs(s)))),
    )
    .sort_values(
        ["mean_osr_macro_f1", "min_osr_macro_f1", "mean_unknown_f1"],
        ascending=False,
    )
)

print("Main deployable VarMax winner:")
display(main_legit_df)

print("Structure-aware VarMax winner:")
display(aligned_df)

Main deployable VarMax winner:


,backbone_tag,mean_osr_macro_f1,min_osr_macro_f1,mean_unknown_f1,min_unknown_f1,mean_unknown_recall,max_abs_bias_delta
0,ref_base_ent005,0.839818,0.660645,0.831886,0.489451,0.798529,0.558824
2,ref_pms_drop040,0.788508,0.570430,0.712335,0.250396,0.662255,0.689542
1,ref_base_lr2e4,0.771503,0.625804,0.742008,0.432815,0.681863,0.537037


Structure-aware VarMax winner:


,backbone_tag,mean_osr_macro_f1,min_osr_macro_f1,mean_unknown_f1,min_unknown_f1,mean_unknown_recall,max_abs_bias_delta
0,ref_base_ent005,0.835928,0.641127,0.801708,0.376361,0.760784,0.718301
1,ref_base_lr2e4,0.791083,0.702175,0.789078,0.625935,0.729902,0.355991
2,ref_pms_drop040,0.748574,0.578178,0.662274,0.317979,0.567157,0.582789


In [3]:
DATA_ROOT = os.path.expanduser("~/Adam/varMax/PADataset/data")
device = "cuda" if torch.cuda.is_available() else "cpu"

refined_df = pd.read_csv("confmanifold_refined_leaderboard.csv")
refined_df["backbone_tag"] = refined_df["run_name"].str.replace(
    r"_unkPA\d+_c\d+_seed\d+$", "", regex=True
)

top_backbone_tags = [
    "ref_base_lr2e4",
    "ref_pms_drop040",
    "ref_base_ent005",
]

runs_to_eval = refined_df[refined_df["backbone_tag"].isin(top_backbone_tags)].copy()
runs_to_eval = runs_to_eval.sort_values(["backbone_tag", "run_name"]).reset_index(drop=True)

display(
    runs_to_eval[
        [
            "backbone_tag",
            "run_name",
            "save_dir",
            "unknown_pas",
            "test_dqn_proxy_softmax3",
            "test_dqn_proxy_expanded5",
            "test_known_macro_f1",
        ]
    ]
)

,backbone_tag,run_name,save_dir,unknown_pas,test_dqn_proxy_softmax3,test_dqn_proxy_expanded5,test_known_macro_f1
0,ref_base_ent005,ref_base_ent005_unkPA2_c16384_seed0,results_pa_confmanifold_refined/22-35_04-29-26...,['PA2'],0.969339,0.914038,0.996296
1,ref_base_ent005,ref_base_ent005_unkPA3_c16384_seed0,results_pa_confmanifold_refined/22-38_04-29-26...,['PA3'],0.987690,0.944207,0.988886
2,ref_base_ent005,ref_base_ent005_unkPA4_c16384_seed0,results_pa_confmanifold_refined/22-41_04-29-26...,['PA4'],0.946951,0.867263,0.963386
3,ref_base_ent005,ref_base_ent005_unkPA8_c16384_seed0,results_pa_confmanifold_refined/22-43_04-29-26...,['PA8'],0.988518,0.987758,0.985178
4,ref_base_lr2e4,ref_base_lr2e4_unkPA2_c16384_seed0,results_pa_confmanifold_refined/22-57_04-29-26...,['PA2'],0.980948,0.892330,0.996296
5,ref_base_lr2e4,ref_base_lr2e4_unkPA3_c16384_seed0,results_pa_confmanifold_refined/23-00_04-29-26...,['PA3'],0.992160,0.942151,0.996296
6,ref_base_lr2e4,ref_base_lr2e4_unkPA4_c16384_seed0,results_pa_confmanifold_refined/23-04_04-29-26...,['PA4'],0.980775,0.923598,0.992592
7,ref_base_lr2e4,ref_base_lr2e4_unkPA8_c16384_seed0,results_pa_confmanifold_refined/23-07_04-29-26...,['PA8'],0.988998,0.952102,0.985178
8,ref_pms_drop040,ref_pms_drop040_unkPA2_c16384_seed0,results_pa_confmanifold_refined/23-45_04-29-26...,['PA2'],0.955565,0.901787,0.996296
9,ref_pms_drop040,ref_pms_drop040_unkPA3_c16384_seed0,results_pa_confmanifold_refined/23-49_04-29-26...,['PA3'],0.996146,0.948451,0.996296


In [4]:
def make_varmax():
    return VarMaxOSR(
        temperature=1.0,
        top2_threshold=None,
        top2_percentile=5.0,
        var_percentiles=(5.0, 95.0),
        energy_percentiles=(5.0, 95.0),
        use_energy=True,
        fit_on="predicted_class",
    )

def make_dqn():
    return DQNOSR(
        state_mode="expanded5",
        gamma=0.95,
        epsilon=1.0,
        epsilon_min=0.05,
        epsilon_decay=0.99,
        learning_rate=1e-3,
        memory_size=2000,
        batch_size=32,
        episodes=30,
        anchor_fraction=0.05,
        train_subsample_size=1250,
        centroid_update_threshold=0.75,
        energy_temperature=1.0,
        seed=42,
        device=device,
    )

def build_varmax_calibration(payload, extras):
    known_cal, open_cal = choose_osr_calibration_splits(
        payload, extras, prefer_balanced=True
    )
    return {
        "mode": "sweep",
        "calibration_mode": "oracle",
        "calibration_known": known_cal,
        "calibration_open": open_cal,
        "known_floor_ratio": 0.95,
        "unknown_recall_floor": 0.60,
        "top_k": 10,
    }

def build_dqn_calibration(payload, extras):
    known_cal, open_cal = choose_osr_calibration_splits(
        payload, extras, prefer_balanced=True
    )
    return {
        "calibration_known": known_cal,
        "calibration_open": open_cal,
    }

method_specs = [
    {
        "name": "VarMax",
        "factory": make_varmax,
        "calibration_builder": build_varmax_calibration,
    },
    {
        "name": "DQN",
        "factory": make_dqn,
        "calibration_builder": build_dqn_calibration,
    },
]

In [5]:
run_row = runs_to_eval.iloc[0]

result = evaluate_multiple_osr_methods_on_run(
    run_dir=run_row["save_dir"],
    method_specs=method_specs,
    checkpoint_tag="best_model",
    batch_size=64,
    num_workers=0,
    pin_memory=True,
    device=device,
    data_root=DATA_ROOT,
)

print(result["handle"].run_name)
print(result["payload"].meta)
print("val_known:", result["payload"].val_known.logits.shape)
print("test_known:", result["payload"].test_known.logits.shape)
print("test_open:", result["payload"].test_open.logits.shape)
print("val_open:", None if result["extras"]["val_open"] is None else result["extras"]["val_open"].logits.shape)
print("val_known_balanced:", None if result["extras"]["val_known_balanced"] is None else result["extras"]["val_known_balanced"].logits.shape)
print("val_open_balanced:", None if result["extras"]["val_open_balanced"] is None else result["extras"]["val_open_balanced"].logits.shape)

display(
    result["df"][
        [
            "method",
            "known_closed_macro_f1",
            "known_osr_macro_f1",
            "unknown_recall",
            "unknown_f1",
            "osr_macro_f1",
            "bias_delta",
            "unknown_auroc",
        ]
    ]
)

ref_base_ent005_unkPA2_c16384_seed0
{'run_name': 'ref_base_ent005_unkPA2_c16384_seed0', 'run_dir': 'results_pa_confmanifold_refined/22-35_04-29-26_ref_base_ent005_unkPA2_c16384_seed0', 'checkpoint_tag': 'best_model', 'unknown_pas': ['PA2'], 'class_names': ['PA3', 'PA4', 'PA8'], 'num_classes': 3, 'cache_len': 16384, 'seed': 0, 'epochs': 200, 'source_type': 'digital', 'source_name': 'pilot_noisy_torch', 'dataset_tag': None, 'noise_tag': None}
val_known: (270, 3)
test_known: (270, 3)
test_open: (510, 3)
val_open: (90, 3)
val_known_balanced: (90, 3)
val_open_balanced: (90, 3)


,method,known_closed_macro_f1,known_osr_macro_f1,unknown_recall,unknown_f1,osr_macro_f1,bias_delta,unknown_auroc
0,VarMax,0.996296,0.902622,0.505882,0.634686,0.687658,0.327451,0.986006
1,DQN,0.996296,0.996296,0.000000,0.000000,0.522273,1.000000,0.908733


In [6]:
all_rows = []

for i, (_, run_row) in enumerate(runs_to_eval.iterrows(), start=1):
    print(f"[{i}/{len(runs_to_eval)}] {run_row['run_name']}")

    result = evaluate_multiple_osr_methods_on_run(
        run_dir=run_row["save_dir"],
        method_specs=method_specs,
        checkpoint_tag="best_model",
        batch_size=64,
        num_workers=0,
        pin_memory=True,
        device=device,
        data_root=DATA_ROOT,
    )

    for row in result["rows"]:
        all_rows.append({
            "backbone_tag": run_row["backbone_tag"],
            "run_name": run_row["run_name"],
            "run_dir": run_row["save_dir"],
            "unknown_pa": run_row["run_name"].split("_unk")[1].split("_")[0],
            "n_val_known_balanced": (
                0 if result["extras"]["val_known_balanced"] is None
                else len(result["extras"]["val_known_balanced"].y_true)
            ),
            "n_val_open_balanced": (
                0 if result["extras"]["val_open_balanced"] is None
                else len(result["extras"]["val_open_balanced"].y_true)
            ),
            **row,
        })

    del result
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

osr_eval_df = pd.DataFrame(all_rows)
osr_eval_df.to_csv("top_backbone_osr_eval.csv", index=False)

display(
    osr_eval_df[
        [
            "backbone_tag",
            "unknown_pa",
            "method",
            "known_osr_macro_f1",
            "unknown_recall",
            "unknown_f1",
            "osr_macro_f1",
            "bias_delta",
            "unknown_auroc",
        ]
    ].sort_values(["backbone_tag", "method", "unknown_pa"])
)

[1/12] ref_base_ent005_unkPA2_c16384_seed0
[2/12] ref_base_ent005_unkPA3_c16384_seed0
[3/12] ref_base_ent005_unkPA4_c16384_seed0
[4/12] ref_base_ent005_unkPA8_c16384_seed0
[5/12] ref_base_lr2e4_unkPA2_c16384_seed0
[6/12] ref_base_lr2e4_unkPA3_c16384_seed0
[7/12] ref_base_lr2e4_unkPA4_c16384_seed0
[8/12] ref_base_lr2e4_unkPA8_c16384_seed0
[9/12] ref_pms_drop040_unkPA2_c16384_seed0
[10/12] ref_pms_drop040_unkPA3_c16384_seed0
[11/12] ref_pms_drop040_unkPA4_c16384_seed0
[12/12] ref_pms_drop040_unkPA8_c16384_seed0


,backbone_tag,unknown_pa,method,known_osr_macro_f1,unknown_recall,unknown_f1,osr_macro_f1,bias_delta,unknown_auroc
1,ref_base_ent005,PA2,DQN,0.996296,0.000000,0.000000,0.522273,1.000000,0.908733
3,ref_base_ent005,PA3,DQN,0.988635,0.454902,0.621984,0.709629,0.530283,0.980581
5,ref_base_ent005,PA4,DQN,0.939961,0.533333,0.677460,0.725621,0.388889,0.855701
7,ref_base_ent005,PA8,DQN,0.985178,0.325490,0.491124,0.630913,0.674510,0.992534
0,ref_base_ent005,PA2,VarMax,0.902622,0.505882,0.634686,0.687658,0.327451,0.986006
2,ref_base_ent005,PA3,VarMax,0.960716,0.890196,0.924644,0.887624,0.043137,0.968475
4,ref_base_ent005,PA4,VarMax,0.971823,0.968627,0.977250,0.952065,0.005447,0.966376
6,ref_base_ent005,PA8,VarMax,0.965570,0.949020,0.964143,0.934001,0.013943,0.987778
9,ref_base_lr2e4,PA2,DQN,0.996296,0.000000,0.000000,0.553851,1.000000,0.852821
11,ref_base_lr2e4,PA3,DQN,0.996296,0.894118,0.944099,0.925555,0.105882,0.996311


In [12]:
RESULTS_ROOT = os.path.expanduser("~/Adam/varMax/PADataset/results_pa_osr_bank")

registry = build_run_registry(RESULTS_ROOT)
print(f"n_runs = {len(registry)}")

n_runs = 48


In [13]:
rows = []
for r in registry:
    rows.append({
        "run_name": r.run_name,
        "split_mode": r.split_mode,
        "unknown_pas": ",".join(r.unknown_pas),
        "cache_len": r.cache_len,
        "batch_size": r.batch_size,
        "lr": r.lr,
        "lambda_center": r.lambda_center,
        "epochs": r.epochs,
        "seed": r.seed,
        "checkpoints": ",".join(list_checkpoint_tags(r)),
    })

registry_df = pd.DataFrame(rows)
display(registry_df.sort_values(["unknown_pas", "cache_len", "seed"]))

,run_name,split_mode,unknown_pas,cache_len,batch_size,lr,lambda_center,epochs,seed,checkpoints
0,osrbank_unkPA2_c4096_bs16_lr5e4_lc01_wd0_seed0,open_pa,PA2,4096,16,0.0005,0.1,100,0,"best_model,epoch_20,epoch_50,epoch_100,final_m..."
1,osrbank_unkPA2_c4096_bs16_lr5e4_lc01_wd0_seed1,open_pa,PA2,4096,16,0.0005,0.1,100,1,"best_model,epoch_20,epoch_50,epoch_100,final_m..."
2,osrbank_unkPA2_c4096_bs16_lr5e4_lc01_wd0_seed2,open_pa,PA2,4096,16,0.0005,0.1,100,2,"best_model,epoch_20,epoch_50,epoch_100,final_m..."
3,osrbank_unkPA2_c8192_bs16_lr5e4_lc01_wd0_seed0,open_pa,PA2,8192,16,0.0005,0.1,100,0,"best_model,epoch_20,epoch_50,epoch_100,final_m..."
4,osrbank_unkPA2_c8192_bs16_lr5e4_lc01_wd0_seed1,open_pa,PA2,8192,16,0.0005,0.1,100,1,"best_model,epoch_20,epoch_50,epoch_100,final_m..."
5,osrbank_unkPA2_c8192_bs16_lr5e4_lc01_wd0_seed2,open_pa,PA2,8192,16,0.0005,0.1,100,2,"best_model,epoch_20,epoch_50,epoch_100,final_m..."
6,osrbank_unkPA2_c16384_bs16_lr5e4_lc01_wd0_seed0,open_pa,PA2,16384,16,0.0005,0.1,100,0,"best_model,epoch_20,epoch_50,epoch_100,final_m..."
7,osrbank_unkPA2_c16384_bs16_lr5e4_lc01_wd0_seed1,open_pa,PA2,16384,16,0.0005,0.1,100,1,"best_model,epoch_20,epoch_50,epoch_100,final_m..."
8,osrbank_unkPA2_c16384_bs16_lr5e4_lc01_wd0_seed2,open_pa,PA2,16384,16,0.0005,0.1,100,2,"best_model,epoch_20,epoch_50,epoch_100,final_m..."
9,osrbank_unkPA2_c32768_bs8_lr5e4_lc01_wd0_seed0,open_pa,PA2,32768,8,0.0005,0.1,100,0,"best_model,epoch_20,epoch_50,epoch_100,final_m..."


In [14]:
candidate_runs = filter_runs(
    registry,
    split_mode="open_pa",
    unknown_pa="PA8",
    cache_len=16384,
    seed=0,
)

print(f"matches = {len(candidate_runs)}")
run = candidate_runs[0]
run

matches = 1


RunRecord(run_name='osrbank_unkPA8_c16384_bs16_lr5e4_lc01_wd0_seed0', run_dir='/home/atrott/Adam/varMax/PADataset/results_pa_osr_bank/18-20_04-22-26_osrbank_unkPA8_c16384_bs16_lr5e4_lc01_wd0_seed0', config_path='/home/atrott/Adam/varMax/PADataset/results_pa_osr_bank/18-20_04-22-26_osrbank_unkPA8_c16384_bs16_lr5e4_lc01_wd0_seed0/config.json', config={'run_name': 'osrbank_unkPA8_c16384_bs16_lr5e4_lc01_wd0_seed0', 'task': 'pa', 'split_mode': 'open_pa', 'unknown_pas': ['PA8'], 'cache_len': 16384, 'batch_size': 16, 'lr': 0.0005, 'weight_decay': 0.0, 'lambda_center': 0.1, 'epochs': 100, 'early_stopping_patience': None, 'seed': 0, 'save_root': 'results_pa_osr_bank'}, task='pa', split_mode='open_pa', unknown_pas=['PA8'], cache_len=16384, batch_size=16, lr=0.0005, weight_decay=0.0, lambda_center=0.1, epochs=100, seed=0, checkpoints={'best_model': CheckpointRecord(tag='best_model', path='/home/atrott/Adam/varMax/PADataset/results_pa_osr_bank/18-20_04-22-26_osrbank_unkPA8_c16384_bs16_lr5e4_lc01_w

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"

handle = load_backbone_run(
    run_dir=run.run_dir,
    checkpoint_tag="best_model",
    device=device,
)

print(handle.run_name)
print(handle.class_names)
print(handle.unknown_pas)

osrbank_unkPA8_c16384_bs16_lr5e4_lc01_wd0_seed0
['PA2', 'PA3', 'PA4']
['PA8']


In [16]:
payload = build_backbone_payload(
    handle,
    include=("val_known", "test_known", "test_open"),
    batch_size=64,
    num_workers=0,
    pin_memory=True,
    return_sample_meta=False,
)

print(payload.meta)
print("val_known logits:", payload.val_known.logits.shape)
print("test_known logits:", payload.test_known.logits.shape)
print("test_open logits:", payload.test_open.logits.shape)
print("feature dim:", payload.test_known.features.shape[1])

{'run_name': 'osrbank_unkPA8_c16384_bs16_lr5e4_lc01_wd0_seed0', 'run_dir': '/home/atrott/Adam/varMax/PADataset/results_pa_osr_bank/18-20_04-22-26_osrbank_unkPA8_c16384_bs16_lr5e4_lc01_wd0_seed0', 'checkpoint_tag': 'best_model', 'unknown_pas': ['PA8'], 'class_names': ['PA2', 'PA3', 'PA4'], 'num_classes': 3, 'cache_len': 16384, 'seed': 0, 'epochs': 100, 'source_type': 'digital', 'source_name': 'pilot_noisy_torch', 'dataset_tag': None, 'noise_tag': None}
val_known logits: (540, 3)
test_known logits: (540, 3)
test_open logits: (1200, 3)
feature dim: 256


In [9]:
from varmax_osr import VarMaxOSR

varmax_oracle = VarMaxOSR(
    temperature=1.0,
    use_energy=True,
    fit_on="predicted_class",
)

varmax_oracle.fit(
    payload,
    calibration={
        "mode": "sweep",
        "calibration_mode": "oracle",
        "calibration_known": payload.val_known,
        "calibration_open": payload.test_open,
        "known_floor_ratio": 0.95,
        "unknown_recall_floor": 0.60,
        "top_k": 10,
    },
)

varmax_oracle.get_params()["best_sweep_result"]

{'calibration_mode': 'oracle',
 'regime': 'oracle',
 'spec_name': 'oracle',
 'true_unknown_name': 'PA8',
 'heldout_class_idx': None,
 'heldout_class_name': None,
 'is_structure_aligned': None,
 'top2_threshold': 0.92,
 'var_percentiles': (15.0, 85.0),
 'energy_percentiles': (5.0, 99.0),
 'feasible': True,
 'known_floor_ratio': 0.95,
 'per_class_floor_ratio': 0.95,
 'known_floor_acc': 0.8708333333333332,
 'known_floor_macro_f1': 0.8695767195767196,
 'per_class_floor_recall': {0: 0.95, 1: 0.95, 2: 0.7124999999999999},
 'baseline_known_closed_acc': 0.9166666666666666,
 'baseline_known_closed_macro_f1': 0.9153439153439153,
 'metrics': {'known_closed_acc': 0.9166666666666666,
  'known_closed_macro_f1': 0.9153439153439153,
  'known_closed_per_class_recall': {0: 1.0, 1: 1.0, 2: 0.75},
  'known_osr_acc': 0.8944444444444445,
  'known_osr_macro_f1': 0.9032890237958894,
  'known_reject_rate': 0.022222222222222223,
  'known_osr_per_class_recall': {0: 0.9555555555555556,
   1: 0.9944444444444445,
 

In [8]:
chosen = varmax_oracle
unknown_label = payload.meta["num_classes"]

known_pred = chosen.predict(payload.test_known, unknown_label=unknown_label)
open_pred  = chosen.predict(payload.test_open, unknown_label=unknown_label)

metrics = evaluate_osr_predictions(
    known_split=payload.test_known,
    open_split=payload.test_open,
    known_pred=known_pred,
    open_pred=open_pred,
    known_class_names=payload.meta["class_names"],
    unknown_label_name="unknown",
)

metrics

{'known_closed_acc': 0.9259259259259259,
 'known_closed_macro_f1': 0.9252397191922873,
 'known_closed_per_class_recall': {0: 1.0,
  1: 0.9944444444444445,
  2: 0.7833333333333333},
 'known_osr_acc': 0.8944444444444445,
 'known_osr_macro_f1': 0.908036056770234,
 'known_reject_rate': 0.03148148148148148,
 'known_osr_per_class_recall': {0: 0.9722222222222222,
  1: 0.9555555555555556,
  2: 0.7555555555555555},
 'known_osr_per_class_reject_rate': {0: 0.027777777777777776,
  1: 0.03888888888888889,
  2: 0.027777777777777776},
 'known_osr_min_per_class_recall': 0.7555555555555555,
 'known_osr_max_per_class_reject_rate': 0.03888888888888889,
 'unknown_precision': 0.9805269186712485,
 'unknown_recall': 0.7133333333333334,
 'unknown_f1': 0.8258562469850458,
 'known_accept_rate': 0.9685185185185186,
 'unknown_detect_rate': 0.7133333333333334,
 'bias_delta': 0.2551851851851852,
 'osr_acc': 0.7695402298850574,
 'osr_macro_f1': 0.7812498408702334,
 'osr_confusion_matrix': array([[175,   0,   0,   5]

In [9]:
varmax_aligned = VarMaxOSR(
    temperature=1.0,
    use_energy=True,
    fit_on="predicted_class",
)

varmax_aligned.fit(
    payload,
    calibration={
        "mode": "sweep",
        "calibration_mode": "surrogate_aligned",
        "calibration_known": payload.val_known,
        "known_floor_ratio": 0.95,
        "per_class_floor_ratio": 0.95,
        "unknown_recall_floor": 0.60,
        "surrogate_fit_frac": 0.50,
        "surrogate_guard_frac": 0.25,
        "surrogate_seed": 0,
        "top_k": 10,
    },
)

varmax_aligned.get_params()["best_sweep_result"]

{'calibration_mode': 'surrogate_aligned',
 'regime': 'aligned',
 'spec_name': 'aligned:PA2',
 'true_unknown_name': 'PA8',
 'heldout_class_idx': 0,
 'heldout_class_name': 'PA2',
 'is_structure_aligned': True,
 'top2_threshold': 0.95,
 'var_percentiles': (10.0, 99.0),
 'energy_percentiles': (10.0, 97.5),
 'feasible': False,
 'known_floor_ratio': 0.95,
 'per_class_floor_ratio': 0.95,
 'known_floor_acc': 0.8708333333333332,
 'known_floor_macro_f1': 0.8695767195767196,
 'per_class_floor_recall': {0: 0.95, 1: 0.95, 2: 0.6333333333333333},
 'baseline_known_closed_acc': 0.9166666666666666,
 'baseline_known_closed_macro_f1': 0.9153439153439153,
 'metrics': {'known_closed_acc': 0.8888888888888888,
  'known_closed_macro_f1': 0.8857142857142858,
  'known_closed_per_class_recall': {0: 1.0, 1: 1.0, 2: 0.6666666666666666},
  'known_osr_acc': 0.8444444444444444,
  'known_osr_macro_f1': 0.8612612612612613,
  'known_reject_rate': 0.044444444444444446,
  'known_osr_per_class_recall': {0: 0.88888888888888

In [10]:
chosen = varmax_aligned
unknown_label = payload.meta["num_classes"]

known_pred = chosen.predict(payload.test_known, unknown_label=unknown_label)
open_pred  = chosen.predict(payload.test_open, unknown_label=unknown_label)

metrics = evaluate_osr_predictions(
    known_split=payload.test_known,
    open_split=payload.test_open,
    known_pred=known_pred,
    open_pred=open_pred,
    known_class_names=payload.meta["class_names"],
    unknown_label_name="unknown",
)

metrics

{'known_closed_acc': 0.9259259259259259,
 'known_closed_macro_f1': 0.9252397191922873,
 'known_closed_per_class_recall': {0: 1.0,
  1: 0.9944444444444445,
  2: 0.7833333333333333},
 'known_osr_acc': 0.8314814814814815,
 'known_osr_macro_f1': 0.8721906698901211,
 'known_reject_rate': 0.09444444444444444,
 'known_osr_per_class_recall': {0: 0.8222222222222222,
  1: 0.9555555555555556,
  2: 0.7166666666666667},
 'known_osr_per_class_reject_rate': {0: 0.17777777777777778,
  1: 0.03888888888888889,
  2: 0.06666666666666667},
 'known_osr_min_per_class_recall': 0.7166666666666667,
 'known_osr_max_per_class_reject_rate': 0.17777777777777778,
 'unknown_precision': 0.9477993858751279,
 'unknown_recall': 0.7716666666666666,
 'unknown_f1': 0.8507119889756546,
 'known_accept_rate': 0.9055555555555556,
 'unknown_detect_rate': 0.7716666666666666,
 'bias_delta': 0.13388888888888895,
 'osr_acc': 0.7902298850574713,
 'osr_macro_f1': 0.776073529690001,
 'osr_confusion_matrix': array([[148,   0,   0,  32],

In [11]:
varmax_mismatch = VarMaxOSR(
    temperature=1.0,
    use_energy=True,
    fit_on="predicted_class",
)

varmax_mismatch.fit(
    payload,
    calibration={
        "mode": "sweep",
        "calibration_mode": "surrogate_mismatched",
        "calibration_known": payload.val_known,
        "known_floor_ratio": 0.95,
        "unknown_recall_floor": 0.60,
        "top_k": 10,
    },
)

varmax_mismatch.get_params()["top_feasible_results"][:3]

[]

In [12]:
from osr_core import evaluate_osr_predictions

chosen = varmax_mismatch
unknown_label = payload.meta["num_classes"]

known_pred = chosen.predict(payload.test_known, unknown_label=unknown_label)
open_pred  = chosen.predict(payload.test_open, unknown_label=unknown_label)

metrics = evaluate_osr_predictions(
    known_split=payload.test_known,
    open_split=payload.test_open,
    known_pred=known_pred,
    open_pred=open_pred,
    known_class_names=payload.meta["class_names"],
    unknown_label_name="unknown",
)

metrics

{'known_closed_acc': 0.9259259259259259,
 'known_closed_macro_f1': 0.9252397191922873,
 'known_closed_per_class_recall': {0: 1.0,
  1: 0.9944444444444445,
  2: 0.7833333333333333},
 'known_osr_acc': 0.8703703703703703,
 'known_osr_macro_f1': 0.892064623425992,
 'known_reject_rate': 0.05555555555555555,
 'known_osr_per_class_recall': {0: 1.0,
  1: 0.9166666666666666,
  2: 0.6944444444444444},
 'known_osr_per_class_reject_rate': {0: 0.0,
  1: 0.07777777777777778,
  2: 0.08888888888888889},
 'known_osr_min_per_class_recall': 0.6944444444444444,
 'known_osr_max_per_class_reject_rate': 0.08888888888888889,
 'unknown_precision': 0.9545454545454546,
 'unknown_recall': 0.525,
 'unknown_f1': 0.6774193548387096,
 'known_accept_rate': 0.9444444444444444,
 'unknown_detect_rate': 0.525,
 'bias_delta': 0.4194444444444444,
 'osr_acc': 0.632183908045977,
 'osr_macro_f1': 0.6982201448383651,
 'osr_confusion_matrix': array([[180,   0,   0,   0],
        [  1, 165,   0,  14],
        [ 39,   0, 125,  16]

In [13]:
varmax_all = VarMaxOSR(
    temperature=1.0,
    use_energy=True,
    fit_on="predicted_class",
)

varmax_all.fit(
    payload,
    calibration={
        "mode": "sweep",
        "calibration_mode": "surrogate_all",
        "calibration_known": payload.val_known,
        "known_floor_ratio": 0.95,
        "unknown_recall_floor": 0.60,
        "top_k": 20,
    },
)

params_all = varmax_all.get_params()
params_all["top_feasible_results"][:5]

[]

In [14]:
from osr_core import evaluate_osr_predictions

chosen = varmax_all
unknown_label = payload.meta["num_classes"]

known_pred = chosen.predict(payload.test_known, unknown_label=unknown_label)
open_pred  = chosen.predict(payload.test_open, unknown_label=unknown_label)

metrics = evaluate_osr_predictions(
    known_split=payload.test_known,
    open_split=payload.test_open,
    known_pred=known_pred,
    open_pred=open_pred,
    known_class_names=payload.meta["class_names"],
    unknown_label_name="unknown",
)

metrics

{'known_closed_acc': 0.9259259259259259,
 'known_closed_macro_f1': 0.9252397191922873,
 'known_closed_per_class_recall': {0: 1.0,
  1: 0.9944444444444445,
  2: 0.7833333333333333},
 'known_osr_acc': 0.8703703703703703,
 'known_osr_macro_f1': 0.892064623425992,
 'known_reject_rate': 0.05555555555555555,
 'known_osr_per_class_recall': {0: 1.0,
  1: 0.9166666666666666,
  2: 0.6944444444444444},
 'known_osr_per_class_reject_rate': {0: 0.0,
  1: 0.07777777777777778,
  2: 0.08888888888888889},
 'known_osr_min_per_class_recall': 0.6944444444444444,
 'known_osr_max_per_class_reject_rate': 0.08888888888888889,
 'unknown_precision': 0.9545454545454546,
 'unknown_recall': 0.525,
 'unknown_f1': 0.6774193548387096,
 'known_accept_rate': 0.9444444444444444,
 'unknown_detect_rate': 0.525,
 'bias_delta': 0.4194444444444444,
 'osr_acc': 0.632183908045977,
 'osr_macro_f1': 0.6982201448383651,
 'osr_confusion_matrix': array([[180,   0,   0,   0],
        [  1, 165,   0,  14],
        [ 39,   0, 125,  16]

In [15]:
dqn_oracle = DQNOSR(
    gamma=0.95,
    epsilon=1.0,
    epsilon_min=0.05,
    epsilon_decay=0.99,
    learning_rate=1e-3,
    memory_size=2000,
    batch_size=32,
    episodes=30,
    anchor_fraction=0.05,
    train_subsample_size=1250,
    centroid_update_threshold=0.75,
    seed=42,
    device="cpu",   # tiny network; faithful and simple
)

dqn_oracle.fit(
    payload,
    calibration={
        "calibration_known": payload.val_known,
        "calibration_open": payload.test_open,
    },
)

dqn_oracle.get_params()["last_fit_summary"]

{'n_states_total': 1740,
 'n_anchor_high': 87,
 'n_anchor_low': 87,
 'n_train_states': 1250,
 'episodes': 30,
 'final_epsilon': 0.05,
 'centroid_known': [0.9357990622520447,
  0.8994137644767761,
  0.2662334144115448],
 'centroid_unknown': [0.6869889497756958,
  0.4744933545589447,
  0.7967000603675842]}

In [16]:
chosen = dqn_oracle
unknown_label = payload.meta["num_classes"]

known_pred = chosen.predict(payload.test_known, unknown_label=unknown_label)
open_pred  = chosen.predict(payload.test_open, unknown_label=unknown_label)

metrics = evaluate_osr_predictions(
    known_split=payload.test_known,
    open_split=payload.test_open,
    known_pred=known_pred,
    open_pred=open_pred,
    known_class_names=payload.meta["class_names"],
    unknown_label_name="unknown",
)

metrics

{'known_closed_acc': 0.9259259259259259,
 'known_closed_macro_f1': 0.9252397191922873,
 'known_closed_per_class_recall': {0: 1.0,
  1: 0.9944444444444445,
  2: 0.7833333333333333},
 'known_osr_acc': 0.9259259259259259,
 'known_osr_macro_f1': 0.9252397191922873,
 'known_reject_rate': 0.0,
 'known_osr_per_class_recall': {0: 1.0,
  1: 0.9944444444444445,
  2: 0.7833333333333333},
 'known_osr_per_class_reject_rate': {0: 0.0, 1: 0.0, 2: 0.0},
 'known_osr_min_per_class_recall': 0.7833333333333333,
 'known_osr_max_per_class_reject_rate': 0.0,
 'unknown_precision': 1.0,
 'unknown_recall': 0.4266666666666667,
 'unknown_f1': 0.5981308411214953,
 'known_accept_rate': 1.0,
 'unknown_detect_rate': 0.4266666666666667,
 'bias_delta': 0.5733333333333333,
 'osr_acc': 0.5816091954022988,
 'osr_macro_f1': 0.6940418933645413,
 'osr_confusion_matrix': array([[180,   0,   0,   0],
        [  1, 179,   0,   0],
        [ 39,   0, 141,   0],
        [676,  12,   0, 512]]),
 'label_names_with_unknown': ['PA2',

In [27]:
varmax_metrics = {
    "known_osr_acc": 0.8944444444444445,
    "known_osr_macro_f1": 0.908036056770234,
    "unknown_precision": 0.9805269186712485,
    "unknown_recall": 0.7133333333333334,
    "unknown_f1": 0.8258562469850458,
    "osr_macro_f1": 0.7812498408702334,
    "bias_delta": 0.2551851851851852,
    "unknown_auroc": 0.7719444444444444,
}

dqn_metrics = {
    "known_osr_acc": 0.9259259259259259,
    "known_osr_macro_f1": 0.9252397191922873,
    "unknown_precision": 1.0,
    "unknown_recall": 0.4266666666666667,
    "unknown_f1": 0.5981308411214953,
    "osr_macro_f1": 0.6940418933645413,
    "bias_delta": 0.5733333333333333,
    "unknown_auroc": 0.7961774691358025,
}

comparison_df = pd.DataFrame([
    {"method": "VarMax-oracle", **varmax_metrics},
    {"method": "DQN-oracle", **dqn_metrics},
])

comparison_df

,method,known_osr_acc,known_osr_macro_f1,unknown_precision,unknown_recall,unknown_f1,osr_macro_f1,bias_delta,unknown_auroc
0,VarMax-oracle,0.894444,0.908036,0.980527,0.713333,0.825856,0.781250,0.255185,0.771944
1,DQN-oracle,0.925926,0.925240,1.000000,0.426667,0.598131,0.694042,0.573333,0.796177


In [19]:
import pandas as pd

comparison_df = pd.DataFrame([
    {"method": "VarMax", **varmax_metrics},   # replace with your saved varmax metrics dict
    {"method": "DQN",    **metrics},
])

comparison_df[[
    "method",
    "known_osr_acc",
    "known_osr_macro_f1",
    "unknown_precision",
    "unknown_recall",
    "unknown_f1",
    "osr_macro_f1",
    "bias_delta",
    "unknown_auroc",
]]

,method,known_osr_acc,known_osr_macro_f1,unknown_precision,unknown_recall,unknown_f1,osr_macro_f1,bias_delta,unknown_auroc
0,VarMax,0.894444,0.908036,0.980527,0.713333,0.825856,0.781250,0.255185,0.771944
1,DQN,0.925926,0.925240,1.000000,0.426667,0.598131,0.694042,0.573333,0.796177


In [21]:
import os
import pandas as pd
import torch

from varmax_osr import VarMaxOSR
from dqn_osr import DQNOSR
from osr_core import (
    build_run_registry,
    filter_runs,
    load_backbone_run,
    build_backbone_payload,
    evaluate_osr_predictions,
)

RESULTS_ROOT = os.path.expanduser("~/Adam/varMax/PADataset/results_pa_osr_bank")


def get_single_run(registry, unknown_pa, cache_len=16384, seed=0):
    matches = filter_runs(
        registry,
        split_mode="open_pa",
        unknown_pa=unknown_pa,
        cache_len=cache_len,
        seed=seed,
    )
    if len(matches) != 1:
        raise RuntimeError(f"Expected 1 run for unknown_pa={unknown_pa}, got {len(matches)}")
    return matches[0]


def eval_varmax_oracle(payload):
    method = VarMaxOSR(
        temperature=1.0,
        use_energy=True,
        fit_on="predicted_class",
    )
    method.fit(
        payload,
        calibration={
            "mode": "sweep",
            "calibration_mode": "oracle",
            "calibration_known": payload.val_known,
            "calibration_open": payload.test_open,
            "known_floor_ratio": 0.95,
            "unknown_recall_floor": 0.60,
            "top_k": 10,
        },
    )

    unknown_label = payload.meta["num_classes"]
    known_pred = method.predict(payload.test_known, unknown_label=unknown_label)
    open_pred = method.predict(payload.test_open, unknown_label=unknown_label)

    metrics = evaluate_osr_predictions(
        known_split=payload.test_known,
        open_split=payload.test_open,
        known_pred=known_pred,
        open_pred=open_pred,
        known_class_names=payload.meta["class_names"],
        unknown_label_name="unknown",
    )
    return method, metrics


def eval_dqn_oracle(payload):
    method = DQNOSR(
        gamma=0.95,
        epsilon=1.0,
        epsilon_min=0.05,
        epsilon_decay=0.99,
        learning_rate=1e-3,
        memory_size=2000,
        batch_size=32,
        episodes=30,
        anchor_fraction=0.05,
        train_subsample_size=1250,
        centroid_update_threshold=0.75,
        seed=42,
        device="cpu",
    )
    method.fit(
        payload,
        calibration={
            "calibration_known": payload.val_known,
            "calibration_open": payload.test_open,
        },
    )

    unknown_label = payload.meta["num_classes"]
    known_pred = method.predict(payload.test_known, unknown_label=unknown_label)
    open_pred = method.predict(payload.test_open, unknown_label=unknown_label)

    metrics = evaluate_osr_predictions(
        known_split=payload.test_known,
        open_split=payload.test_open,
        known_pred=known_pred,
        open_pred=open_pred,
        known_class_names=payload.meta["class_names"],
        unknown_label_name="unknown",
    )
    return method, metrics


registry = build_run_registry(RESULTS_ROOT)
device = "cuda" if torch.cuda.is_available() else "cpu"

rows = []

for unknown_pa in ["PA2", "PA3", "PA4", "PA8"]:
    run = get_single_run(registry, unknown_pa=unknown_pa, cache_len=16384, seed=0)

    handle = load_backbone_run(
        run_dir=run.run_dir,
        checkpoint_tag="best_model",
        device=device,
    )

    payload = build_backbone_payload(
        handle,
        include=("val_known", "test_known", "test_open"),
        batch_size=64,
        num_workers=0,
        pin_memory=True,
        return_sample_meta=False,
    )

    _, varmax_metrics = eval_varmax_oracle(payload)
    _, dqn_metrics = eval_dqn_oracle(payload)

    for method_name, m in [("VarMax-oracle", varmax_metrics), ("DQN-oracle", dqn_metrics)]:
        rows.append({
            "unknown_pa": unknown_pa,
            "method": method_name,
            "known_osr_acc": m["known_osr_acc"],
            "known_osr_macro_f1": m["known_osr_macro_f1"],
            "unknown_precision": m["unknown_precision"],
            "unknown_recall": m["unknown_recall"],
            "unknown_f1": m["unknown_f1"],
            "osr_macro_f1": m["osr_macro_f1"],
            "bias_delta": m["bias_delta"],
            "unknown_auroc": m["unknown_auroc"],
        })

all_folds_df = pd.DataFrame(rows)
all_folds_df.sort_values(["unknown_pa", "method"])

,unknown_pa,method,known_osr_acc,known_osr_macro_f1,unknown_precision,unknown_recall,unknown_f1,osr_macro_f1,bias_delta,unknown_auroc
1,PA2,DQN-oracle,0.605556,0.635032,0.857143,0.905000,0.880422,0.695561,-0.240185,0.818636
0,PA2,VarMax-oracle,0.940741,0.940269,0.000000,0.000000,0.000000,0.532142,1.000000,0.814659
3,PA3,DQN-oracle,0.911111,0.925000,0.944444,0.226667,0.365591,0.613736,0.743704,0.894498
2,PA3,VarMax-oracle,0.924074,0.931275,0.991604,0.885833,0.935739,0.864208,0.097500,0.971112
5,PA4,DQN-oracle,0.648148,0.656268,0.794788,0.610000,0.690240,0.469845,0.040000,0.694657
4,PA4,VarMax-oracle,0.970370,0.982186,0.984049,0.668333,0.796030,0.762566,0.307593,0.731724
7,PA8,DQN-oracle,0.925926,0.925240,1.000000,0.426667,0.598131,0.694042,0.573333,0.796177
6,PA8,VarMax-oracle,0.894444,0.908036,0.980527,0.713333,0.825856,0.781250,0.255185,0.771944


In [10]:
all_folds_df.pivot(
    index="unknown_pa",
    columns="method",
    values=["unknown_recall", "unknown_f1", "osr_macro_f1", "unknown_auroc", "bias_delta"]
)

NameError: name 'all_folds_df' is not defined

In [31]:
import os
import numpy as np
import pandas as pd
import torch

from varmax_osr import VarMaxOSR
from dqn_osr import DQNOSR
from osr_core import (
    build_run_registry,
    filter_runs,
    load_backbone_run,
    build_backbone_payload,
    evaluate_osr_predictions,
    SplitOutputs,
)

RESULTS_ROOT = os.path.expanduser("~/Adam/varMax/PADataset/results_pa_osr_bank")


def subset_split(split: SplitOutputs, idx: np.ndarray, split_name: str = None) -> SplitOutputs:
    idx = np.asarray(idx)
    if idx.dtype == bool:
        idx = np.where(idx)[0]

    sample_meta = None
    if split.sample_meta is not None:
        sample_meta = [split.sample_meta[int(i)] for i in idx]

    return SplitOutputs(
        split_name=split_name or split.split_name,
        y_true=split.y_true[idx],
        logits=split.logits[idx],
        features=split.features[idx],
        closed_pred=split.closed_pred[idx],
        probs=split.probs[idx],
        sample_meta=sample_meta,
    )


def make_balanced_oracle_calibration(payload, seed=42):
    known = payload.val_known
    open_full = payload.test_open

    n_known = len(known.y_true)
    n_open = len(open_full.y_true)
    if n_open < n_known:
        raise ValueError(f"Not enough open samples: n_open={n_open}, n_known={n_known}")

    rng = np.random.default_rng(seed)
    open_idx = rng.choice(n_open, size=n_known, replace=False)

    open_bal = subset_split(
        open_full,
        open_idx,
        split_name=f"{open_full.split_name}_balanced",
    )
    return known, open_bal


def get_single_run(registry, unknown_pa, cache_len=16384, seed=0):
    matches = filter_runs(
        registry,
        split_mode="open_pa",
        unknown_pa=unknown_pa,
        cache_len=cache_len,
        seed=seed,
    )
    if len(matches) != 1:
        raise RuntimeError(f"Expected 1 run for unknown_pa={unknown_pa}, got {len(matches)}")
    return matches[0]


def eval_varmax_oracle(payload):
    method = VarMaxOSR(
        temperature=1.0,
        use_energy=True,
        fit_on="predicted_class",
    )
    method.fit(
        payload,
        calibration={
            "mode": "sweep",
            "calibration_mode": "oracle",
            "calibration_known": payload.val_known,
            "calibration_open": payload.test_open,
            "known_floor_ratio": 0.95,
            "unknown_recall_floor": 0.60,
            "top_k": 10,
        },
    )

    unknown_label = payload.meta["num_classes"]
    known_pred = method.predict(payload.test_known, unknown_label=unknown_label)
    open_pred = method.predict(payload.test_open, unknown_label=unknown_label)

    metrics = evaluate_osr_predictions(
        known_split=payload.test_known,
        open_split=payload.test_open,
        known_pred=known_pred,
        open_pred=open_pred,
        known_class_names=payload.meta["class_names"],
        unknown_label_name="unknown",
    )
    return method, metrics


def eval_dqn_oracle_balanced_expanded(payload, seed=42):
    cal_known_bal, cal_open_bal = make_balanced_oracle_calibration(payload, seed=seed)

    method = DQNOSR(
        state_mode="expanded5",
        gamma=0.95,
        epsilon=1.0,
        epsilon_min=0.05,
        epsilon_decay=0.99,
        learning_rate=1e-3,
        memory_size=2000,
        batch_size=32,
        episodes=30,
        anchor_fraction=0.05,
        train_subsample_size=1250,
        centroid_update_threshold=0.75,
        energy_temperature=1.0,
        seed=seed,
        device="cpu",
    )
    method.fit(
        payload,
        calibration={
            "calibration_known": cal_known_bal,
            "calibration_open": cal_open_bal,
        },
    )

    unknown_label = payload.meta["num_classes"]
    known_pred = method.predict(payload.test_known, unknown_label=unknown_label)
    open_pred = method.predict(payload.test_open, unknown_label=unknown_label)

    metrics = evaluate_osr_predictions(
        known_split=payload.test_known,
        open_split=payload.test_open,
        known_pred=known_pred,
        open_pred=open_pred,
        known_class_names=payload.meta["class_names"],
        unknown_label_name="unknown",
    )
    return method, metrics

In [32]:
cal_known_bal, cal_open_bal = make_balanced_oracle_calibration(payload, seed=42)

print("known calib size:", len(cal_known_bal.y_true))
print("open calib size :", len(cal_open_bal.y_true))

known calib size: 540
open calib size : 540


In [22]:
dqn_oracle_bal = DQNOSR(
    gamma=0.95,
    epsilon=1.0,
    epsilon_min=0.05,
    epsilon_decay=0.99,
    learning_rate=1e-3,
    memory_size=2000,
    batch_size=32,
    episodes=30,
    anchor_fraction=0.05,
    train_subsample_size=1250,
    centroid_update_threshold=0.75,
    seed=42,
    device="cpu",
)

dqn_oracle_bal.fit(
    payload,
    calibration={
        "calibration_known": cal_known_bal,
        "calibration_open": cal_open_bal,
    },
)

dqn_oracle_bal.get_params()["last_fit_summary"]

{'n_states_total': 1080,
 'n_anchor_high': 54,
 'n_anchor_low': 54,
 'n_train_states': 972,
 'episodes': 30,
 'final_epsilon': 0.05,
 'state_mode': 'softmax3',
 'state_size': 3,
 'centroid_known': [0.9411777853965759,
  0.9075965285301208,
  0.24910469353199005],
 'centroid_unknown': [0.7065967917442322,
  0.5127962827682495,
  0.7742995619773865]}

In [29]:
unknown_label = payload.meta["num_classes"]

known_pred_bal = dqn_oracle_bal.predict(payload.test_known, unknown_label=unknown_label)
open_pred_bal  = dqn_oracle_bal.predict(payload.test_open, unknown_label=unknown_label)

dqn_bal_metrics = evaluate_osr_predictions(
    known_split=payload.test_known,
    open_split=payload.test_open,
    known_pred=known_pred_bal,
    open_pred=open_pred_bal,
    known_class_names=payload.meta["class_names"],
    unknown_label_name="unknown",
)

dqn_bal_metrics

{'known_closed_acc': 0.9259259259259259,
 'known_closed_macro_f1': 0.9252397191922873,
 'known_closed_per_class_recall': {0: 1.0,
  1: 0.9944444444444445,
  2: 0.7833333333333333},
 'known_osr_acc': 0.9259259259259259,
 'known_osr_macro_f1': 0.9252397191922873,
 'known_reject_rate': 0.0,
 'known_osr_per_class_recall': {0: 1.0,
  1: 0.9944444444444445,
  2: 0.7833333333333333},
 'known_osr_per_class_reject_rate': {0: 0.0, 1: 0.0, 2: 0.0},
 'known_osr_min_per_class_recall': 0.7833333333333333,
 'known_osr_max_per_class_reject_rate': 0.0,
 'unknown_precision': 1.0,
 'unknown_recall': 0.43333333333333335,
 'unknown_f1': 0.6046511627906976,
 'known_accept_rate': 1.0,
 'unknown_detect_rate': 0.43333333333333335,
 'bias_delta': 0.5666666666666667,
 'osr_acc': 0.5862068965517241,
 'osr_macro_f1': 0.6962985140266102,
 'osr_confusion_matrix': array([[180,   0,   0,   0],
        [  1, 179,   0,   0],
        [ 39,   0, 141,   0],
        [668,  12,   0, 520]]),
 'label_names_with_unknown': ['PA2

In [27]:
comparison_df = pd.DataFrame([
    {"method": "VarMax-oracle", **varmax_metrics},
    {"method": "DQN-oracle-unbalanced", **dqn_metrics},
    {"method": "DQN-oracle-balanced", **dqn_bal_metrics},
])

comparison_df[[
    "method",
    "known_osr_acc",
    "known_osr_macro_f1",
    "unknown_precision",
    "unknown_recall",
    "unknown_f1",
    "osr_macro_f1",
    "bias_delta",
    "unknown_auroc",
]]

,method,known_osr_acc,known_osr_macro_f1,unknown_precision,unknown_recall,unknown_f1,osr_macro_f1,bias_delta,unknown_auroc
0,VarMax-oracle,0.894444,0.908036,0.980527,0.713333,0.825856,0.781250,0.255185,0.771944
1,DQN-oracle-unbalanced,0.925926,0.925240,1.000000,0.426667,0.598131,0.694042,0.573333,0.796177
2,DQN-oracle-balanced,0.925926,0.925240,1.000000,0.433333,0.604651,0.696299,0.566667,0.787790


In [18]:
dqn_oracle_expanded = DQNOSR(
    state_mode="expanded5",
    gamma=0.95,
    epsilon=1.0,
    epsilon_min=0.05,
    epsilon_decay=0.99,
    learning_rate=1e-3,
    memory_size=2000,
    batch_size=32,
    episodes=30,
    anchor_fraction=0.05,
    train_subsample_size=1250,
    centroid_update_threshold=0.75,
    energy_temperature=1.0,
    seed=42,
    device="cpu",
)

In [23]:
dqn_oracle_expanded = DQNOSR(
    state_mode="expanded5",
    gamma=0.95,
    epsilon=1.0,
    epsilon_min=0.05,
    epsilon_decay=0.99,
    learning_rate=1e-3,
    memory_size=2000,
    batch_size=32,
    episodes=30,
    anchor_fraction=0.05,
    train_subsample_size=1250,
    centroid_update_threshold=0.75,
    energy_temperature=1.0,
    seed=42,
    device="cpu",
)

In [24]:
dqn_oracle_expanded.fit(
    payload,
    calibration={
        "calibration_known": cal_known_bal,
        "calibration_open": cal_open_bal,
    },
)

dqn_oracle_expanded.get_params()["last_fit_summary"]

{'n_states_total': 1080,
 'n_anchor_high': 54,
 'n_anchor_low': 54,
 'n_train_states': 972,
 'episodes': 30,
 'final_epsilon': 0.05,
 'state_mode': 'expanded5',
 'state_size': 5,
 'centroid_known': [0.9443713426589966,
  0.9125096201896667,
  0.239656463265419,
  2.4835829734802246,
  0.31524401903152466],
 'centroid_unknown': [0.7234007716178894,
  0.542902946472168,
  0.7457296252250671,
  1.40780770778656,
  0.12074507027864456]}

In [25]:
unknown_label = payload.meta["num_classes"]

known_pred_exp = dqn_oracle_expanded.predict(payload.test_known, unknown_label=unknown_label)
open_pred_exp  = dqn_oracle_expanded.predict(payload.test_open, unknown_label=unknown_label)

dqn_expanded_metrics = evaluate_osr_predictions(
    known_split=payload.test_known,
    open_split=payload.test_open,
    known_pred=known_pred_exp,
    open_pred=open_pred_exp,
    known_class_names=payload.meta["class_names"],
    unknown_label_name="unknown",
)

dqn_expanded_metrics

{'known_closed_acc': 0.9259259259259259,
 'known_closed_macro_f1': 0.9252397191922873,
 'known_closed_per_class_recall': {0: 1.0,
  1: 0.9944444444444445,
  2: 0.7833333333333333},
 'known_osr_acc': 0.9259259259259259,
 'known_osr_macro_f1': 0.9252397191922873,
 'known_reject_rate': 0.0,
 'known_osr_per_class_recall': {0: 1.0,
  1: 0.9944444444444445,
  2: 0.7833333333333333},
 'known_osr_per_class_reject_rate': {0: 0.0, 1: 0.0, 2: 0.0},
 'known_osr_min_per_class_recall': 0.7833333333333333,
 'known_osr_max_per_class_reject_rate': 0.0,
 'unknown_precision': 1.0,
 'unknown_recall': 0.4583333333333333,
 'unknown_f1': 0.6285714285714286,
 'known_accept_rate': 1.0,
 'unknown_detect_rate': 0.4583333333333333,
 'bias_delta': 0.5416666666666667,
 'osr_acc': 0.603448275862069,
 'osr_macro_f1': 0.7105693256357062,
 'osr_confusion_matrix': array([[180,   0,   0,   0],
        [  1, 179,   0,   0],
        [ 39,   0, 141,   0],
        [648,   2,   0, 550]]),
 'label_names_with_unknown': ['PA2', 

In [30]:
comparison_df = pd.DataFrame([
    {"method": "VarMax-oracle", **varmax_metrics},
    {"method": "DQN-oracle-unbalanced", **dqn_metrics},
    {"method": "DQN-oracle-balanced", **dqn_bal_metrics},
    {"method": "DQN-oracle-expanded5", **dqn_expanded_metrics},
])

comparison_df[[
    "method",
    "known_osr_acc",
    "known_osr_macro_f1",
    "unknown_precision",
    "unknown_recall",
    "unknown_f1",
    "osr_macro_f1",
    "bias_delta",
    "unknown_auroc",
]]

,method,known_osr_acc,known_osr_macro_f1,unknown_precision,unknown_recall,unknown_f1,osr_macro_f1,bias_delta,unknown_auroc
0,VarMax-oracle,0.894444,0.908036,0.980527,0.713333,0.825856,0.781250,0.255185,0.771944
1,DQN-oracle-unbalanced,0.925926,0.925240,1.000000,0.426667,0.598131,0.694042,0.573333,0.796177
2,DQN-oracle-balanced,0.925926,0.925240,1.000000,0.433333,0.604651,0.696299,0.566667,0.787790
3,DQN-oracle-expanded5,0.925926,0.925240,1.000000,0.458333,0.628571,0.710569,0.541667,0.777662


In [34]:
registry = build_run_registry(RESULTS_ROOT)
device = "cuda" if torch.cuda.is_available() else "cpu"

rows = []
fit_summaries = []

for unknown_pa in ["PA2", "PA3", "PA4", "PA8"]:
    run = get_single_run(registry, unknown_pa=unknown_pa, cache_len=16384, seed=0)

    handle = load_backbone_run(
        run_dir=run.run_dir,
        checkpoint_tag="best_model",
        device=device,
    )

    payload = build_backbone_payload(
        handle,
        include=("val_known", "test_known", "test_open"),
        batch_size=64,
        num_workers=0,
        pin_memory=True,
        return_sample_meta=False,
    )

    varmax_method, varmax_metrics = eval_varmax_oracle(payload)
    dqn_method, dqn_metrics = eval_dqn_oracle_balanced_expanded(payload, seed=42)

    fit_summaries.append({
        "unknown_pa": unknown_pa,
        "dqn_fit_summary": dqn_method.get_params()["last_fit_summary"],
    })

    for method_name, m in [
        ("VarMax-oracle", varmax_metrics),
        ("DQN-oracle-expanded5-balanced", dqn_metrics),
    ]:
        rows.append({
            "unknown_pa": unknown_pa,
            "method": method_name,
            "known_osr_acc": m["known_osr_acc"],
            "known_osr_macro_f1": m["known_osr_macro_f1"],
            "unknown_precision": m["unknown_precision"],
            "unknown_recall": m["unknown_recall"],
            "unknown_f1": m["unknown_f1"],
            "osr_macro_f1": m["osr_macro_f1"],
            "bias_delta": m["bias_delta"],
            "unknown_auroc": m["unknown_auroc"],
            "known_reject_rate": m["known_reject_rate"],
        })

expanded5_all_folds_df = pd.DataFrame(rows)
expanded5_all_folds_df.sort_values(["unknown_pa", "method"])

,unknown_pa,method,known_osr_acc,known_osr_macro_f1,unknown_precision,unknown_recall,unknown_f1,osr_macro_f1,bias_delta,unknown_auroc,known_reject_rate
1,PA2,DQN-oracle-expanded5-balanced,0.272222,0.299694,0.759814,0.951667,0.844987,0.436017,-0.620185,0.582747,0.668519
0,PA2,VarMax-oracle,0.940741,0.940269,0.000000,0.000000,0.000000,0.532142,1.000000,0.814659,0.000000
3,PA3,DQN-oracle-expanded5-balanced,0.275926,0.303943,0.745751,0.877500,0.806279,0.362620,-0.542315,0.392255,0.664815
2,PA3,VarMax-oracle,0.924074,0.931275,0.991604,0.885833,0.935739,0.864208,0.097500,0.971112,0.016667
5,PA4,DQN-oracle-expanded5-balanced,0.000000,0.000000,0.637037,0.788333,0.704655,0.176164,-0.786481,0.465901,0.998148
4,PA4,VarMax-oracle,0.970370,0.982186,0.984049,0.668333,0.796030,0.762566,0.307593,0.731724,0.024074
7,PA8,DQN-oracle-expanded5-balanced,0.925926,0.925240,1.000000,0.458333,0.628571,0.710569,0.541667,0.777662,0.000000
6,PA8,VarMax-oracle,0.894444,0.908036,0.980527,0.713333,0.825856,0.781250,0.255185,0.771944,0.031481


In [ ]:
expanded5_all_folds_df.pivot(
    index="unknown_pa",
    columns="method",
    values=[
        "unknown_recall",
        "unknown_f1",
        "osr_macro_f1",
        "unknown_auroc",
        "bias_delta",
        "known_reject_rate",
    ],
)

In [15]:
varmax_final_df[
    (varmax_final_df["backbone_tag"] == "ref_base_ent005") &
    (varmax_final_df["method"] == "VarMax-surrogate-all")
][[
    "unknown_pa",
    "best_top2_threshold",
    "best_var_percentiles",
    "best_energy_percentiles",
    "best_regime",
    "best_heldout_class_name",
    "best_is_structure_aligned",
    "known_osr_macro_f1",
    "unknown_recall",
    "unknown_f1",
    "osr_macro_f1",
    "bias_delta",
    "no_feasible_solution"
]]

,unknown_pa,best_top2_threshold,best_var_percentiles,best_energy_percentiles,best_regime,best_heldout_class_name,best_is_structure_aligned,known_osr_macro_f1,unknown_recall,unknown_f1,osr_macro_f1,bias_delta,no_feasible_solution
1,PA2,0.92,"(10.0, 85.0)","(10.0, 95.0)",mismatched,PA3,False,0.942303,0.341176,0.489451,0.660645,0.558824,True
4,PA3,0.94,"(10.0, 85.0)","(15.0, 92.5)",mismatched,PA2,False,0.912483,0.925490,0.920976,0.867893,-0.084749,True
7,PA4,0.96,"(7.5, 85.0)","(5.0, 85.0)",aligned,PA3,True,0.933728,0.968627,0.963902,0.920489,-0.046405,True
10,PA8,0.92,"(15.0, 85.0)","(7.5, 90.0)",aligned,PA2,True,0.929672,0.958824,0.953216,0.910244,-0.058824,True
